# Прогнозирование продаж Favorita

## 1. Постановка задачи

Цель работы - построить модель прогноза ежедневных продаж товаров в магазинах Favorita.

Прогноз требуется для каждой пары store_nbr–item_nbr на 16 дней: с 16.08.2017 по 31.08.2017.

Данные имеют панельную структуру: одновременно наблюдается большое число временных рядов, соответствующих разным товарам и магазинам. Помимо истории продаж доступны сведения об акциях, товарах, магазинах, праздниках и цене нефти.

План работы: последовательно сравнить простые baseline-модели, классические методы прогнозирования, ML- и DL-модели.

Основная валидация использует тот же горизонт, что и соревнование: последние 16 дней train (31.07.2017–15.08.2017) используются как validation.

Для сравнения моделей использую MAE и NWRMSLE. Последняя метрика учитывает логарифмический масштаб ошибки и больший вес скоропортящихся товаров.

## 2. Загрузка данных и первичный анализ

Смотрим на вспомогательные таблицы.

In [3]:
import os

DATA_PATH = (
    '/kaggle/input/datasets/m2101119/'
    'favorita-grocery-sales-forecasting'
)

print(os.listdir(DATA_PATH))

['oil.csv', 'items.csv', 'sample_submission.csv', 'holidays_events.csv', 'stores.csv', 'train.csv', 'test.csv', 'transactions.csv']


In [124]:
import plotly.express as px
import plotly.graph_objects as go

In [5]:
TRAIN_PATH = os.path.join(DATA_PATH, 'train.csv')
TEST_PATH = os.path.join(DATA_PATH, 'test.csv')
ITEMS_PATH = os.path.join(DATA_PATH, 'items.csv')
STORES_PATH = os.path.join(DATA_PATH, 'stores.csv')
OIL_PATH = os.path.join(DATA_PATH, 'oil.csv')
HOLIDAYS_PATH = os.path.join(DATA_PATH, 'holidays_events.csv')
TRANSACTIONS_PATH = os.path.join(DATA_PATH, 'transactions.csv')
SAMPLE_SUBMISSION_PATH = os.path.join(
    DATA_PATH,
    'sample_submission.csv'
)

for path in [
    TRAIN_PATH,
    TEST_PATH,
    ITEMS_PATH,
    STORES_PATH,
    OIL_PATH,
    HOLIDAYS_PATH,
    TRANSACTIONS_PATH,
    SAMPLE_SUBMISSION_PATH
]:
    print(os.path.exists(path), path)

True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/train.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/test.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/items.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/stores.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/oil.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/holidays_events.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/transactions.csv
True /kaggle/input/datasets/m2101119/favorita-grocery-sales-forecasting/sample_submission.csv


In [6]:
import pandas as pd
import numpy as np
import gc

items = pd.read_csv(ITEMS_PATH)
stores = pd.read_csv(STORES_PATH)
oil = pd.read_csv(OIL_PATH, parse_dates=['date'])
holidays = pd.read_csv(HOLIDAYS_PATH, parse_dates=['date'])
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    parse_dates=['date']
)
test = pd.read_csv(
    TEST_PATH,
    parse_dates=['date']
)

print('items:', items.shape)
print('stores:', stores.shape)
print('oil:', oil.shape)
print('holidays:', holidays.shape)
print('transactions:', transactions.shape)
print('test:', test.shape)

items: (4100, 4)
stores: (54, 5)
oil: (1218, 2)
holidays: (350, 6)
transactions: (83488, 3)
test: (3370464, 5)


In [127]:
from pathlib import Path

DATA_PATH = Path(
    '/kaggle/input/datasets/m2101119/'
    'favorita-grocery-sales-forecasting'
)

train_path = DATA_PATH / 'train.csv'

### 2.1 Магазины (stores.csv)

In [128]:
stores = pd.read_csv(DATA_PATH / 'stores.csv')
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [129]:
stores.shape

(54, 5)

In [130]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [131]:
print('Количество уникальных городов:', stores['city'].nunique())
print('\nКоличество магазинов по типам:')
print(stores['type'].value_counts())
print('\nКоличество уникальных кластеров:', stores['cluster'].nunique())
print('\nРаспределение магазинов по кластерам:')
print(stores['cluster'].value_counts().sort_index())


Количество уникальных городов: 22

Количество магазинов по типам:
type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64

Количество уникальных кластеров: 17

Распределение магазинов по кластерам:
cluster
1     3
2     2
3     7
4     3
5     1
6     6
7     2
8     3
9     2
10    6
11    3
12    1
13    4
14    4
15    5
16    1
17    1
Name: count, dtype: int64


В таблице содержится информация о 54 магазинах, расположенных в 22 городах, т.е. существуют города, в которых несколько магазинов.

Спрос между городами может отличаться.

Самый распространенный тип магазинов - D, самый редкий - Е.

Кластеры распределены неравномерно: в 3 кластере больше всего магазинов - 7. Есть ещё два крупных кластера - 6 и 10 - в них по 6 магазинов. Также есть кластеры, в которых только 1 магазин.

Пропусков в таблице нет.

In [132]:
store_type_counts = (
    stores['type']
    .value_counts()
    .reset_index()
)

store_type_counts.columns = [
    'Тип магазина',
    'Количество магазинов'
]

fig = px.bar(
    store_type_counts,
    x = 'Тип магазина',
    y = 'Количество магазинов',
    title = 'Распределение магазинов по типам',
    text = 'Количество магазинов'
)

fig.show()

### 2.2 Товары (items.csv)



In [133]:
items = pd.read_csv(DATA_PATH / 'items.csv')
items.head()


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


In [134]:
items.shape


(4100, 4)

In [135]:
items.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB


In [136]:
print('Количество уникальных family:', items['family'].nunique())
print('\nКоличество товаров по family:')
print(items['family'].value_counts())
print('\nРаспределение perishable:')
print(items['perishable'].value_counts())
print('\nДоля скоропортящихся товаров:')
print(
    items['perishable']
    .value_counts(normalize=True)
    .sort_index()
)

Количество уникальных family: 33

Количество товаров по family:
family
GROCERY I                     1334
BEVERAGES                      613
CLEANING                       446
PRODUCE                        306
DAIRY                          242
PERSONAL CARE                  153
BREAD/BAKERY                   134
HOME CARE                      108
DELI                            91
MEATS                           84
HOME AND KITCHEN I              77
LIQUOR,WINE,BEER                73
FROZEN FOODS                    55
POULTRY                         54
HOME AND KITCHEN II             45
EGGS                            41
CELEBRATION                     31
LAWN AND GARDEN                 26
PREPARED FOODS                  26
LADIESWEAR                      21
LINGERIE                        20
AUTOMOTIVE                      20
BEAUTY                          19
PLAYERS AND ELECTRONICS         17
SCHOOL AND OFFICE SUPPLIES      15
GROCERY II                      14
PET SUPPLIES       

В таблице нет пропущенных значений.

Присутствует бинарный признак perishable, указывающий, относится ли товар к скоропортящимся.

Скоропортящихся товаров около четверти от всех, и спрос на них может быть связан с днём недели, праздниками, акциями и т.д.

Категориальный признак family поможет понять, как ведут себя похожие товары. Это может быть использовано, если в test попадут товары, которых не было в train.

Присутствует несколько маленьких семейств товаров - baby care, books и home appliance (по 1 товару). Модель может хуже работать с этими категориями, поскольку она видит мало примеров.

In [137]:
family_counts = (
    items['family']
    .value_counts()
    .head(15)
    .reset_index()
)

family_counts.columns = [
    'Family',
    'Количество товаров'
]

fig = px.bar(
    family_counts,
    x='Количество товаров',
    y='Family',
    orientation='h',
    title='Топ-15 товарных семейств по количеству товаров',
    text='Количество товаров'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

### 2.3 Цена нефти (oil.csv)

In [138]:
oil = pd.read_csv(DATA_PATH / 'oil.csv', parse_dates=['date'])

In [139]:
oil.head(10)


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


In [140]:
oil.shape


(1218, 2)

In [141]:
oil.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [142]:
print('Начальная дата:', oil['date'].min())
print('Конечная дата:', oil['date'].max())
print('\nКоличество пропусков:')
print(oil.isna().sum())
print('\nДоля пропусков, %:')
print(oil.isna().mean() * 100)

Начальная дата: 2013-01-01 00:00:00
Конечная дата: 2017-08-31 00:00:00

Количество пропусков:
date           0
dcoilwtico    43
dtype: int64

Доля пропусков, %:
date          0.000000
dcoilwtico    3.530378
dtype: float64


Признак dcoilwtico имеет пропуски: записей 1218, а значений обнаружено 1175, т.е. пропущено 43 записи (3,53% от всех). Это небольшая доля, значения можно восстановить.

Посмотрим на даты. Таблица содержит значения не для каждого календарного дня: в частности, отсутствуют многие выходные. Перед использованием признака ряд потребуется привести к ежедневной частоте и заполнить пропуски.

In [143]:
fig = px.line(
    oil,
    x='date',
    y='dcoilwtico',
    title='Динамика цены нефти во времени'
)

fig.update_layout(
    xaxis_title='Дата',
    yaxis_title='Цена нефти'
)

fig.show()

### 2.4 Праздники и события (holidays_events.csv)



In [144]:
holidays = pd.read_csv(DATA_PATH / 'holidays_events.csv', parse_dates=['date'])
print('Размер таблицы:', holidays.shape)
holidays.head()


Размер таблицы: (350, 6)


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [145]:
print('Информация о таблице:')
holidays.info()
print('\nТипы событий:')
print(holidays['type'].value_counts())
print('\nУровни праздников:')
print(holidays['locale'].value_counts())
print('\nПеренесённые праздники:')
print(holidays['transferred'].value_counts())
print('\nКоличество пропусков:')
print(holidays.isna().sum())


Информация о таблице:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4   description  350 non-null    object        
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[ns](1), object(4)
memory usage: 14.1+ KB

Типы событий:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

Уровни праздников:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

Перенесённые праздники:
transferred
False    338
True      12
Name: count, dtype: int64

Количество пропусков:
date           0
type           0
locale      

### 2.5 Транзакции (transactions.csv)

In [146]:
transactions = pd.read_csv(DATA_PATH / 'transactions.csv', parse_dates=['date'])
print('Размер таблицы:', transactions.shape)
transactions.head()


Размер таблицы: (83488, 3)


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [147]:
print('Начальная дата:', transactions['date'].min())
print('Конечная дата:', transactions['date'].max())
print('\nКоличество уникальных магазинов:')
print(transactions['store_nbr'].nunique())
print('\nКоличество пропусков:')
print(transactions.isna().sum())
print('\nСтатистика числа транзакций:')
print(transactions['transactions'].describe())


Начальная дата: 2013-01-01 00:00:00
Конечная дата: 2017-08-15 00:00:00

Количество уникальных магазинов:
54

Количество пропусков:
date            0
store_nbr       0
transactions    0
dtype: int64

Статистика числа транзакций:
count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64


### 2.6 Тестовая выборка (`test.csv`)

In [148]:
test = pd.read_csv(DATA_PATH / 'test.csv', parse_dates=['date'])
print('Размер таблицы:', test.shape)
test.head()


Размер таблицы: (3370464, 5)


,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,False
1,125497041,2017-08-16,1,99197,False
2,125497042,2017-08-16,1,103501,False
3,125497043,2017-08-16,1,103520,False
4,125497044,2017-08-16,1,103665,False


Определим горизонт прогнозирования:

In [149]:
print('Начальная дата test:', test['date'].min())
print('Конечная дата test:', test['date'].max())
print('\nКоличество уникальных дат:')
print(test['date'].nunique())
print('\nКоличество уникальных магазинов:')
print(test['store_nbr'].nunique())
print('\nКоличество уникальных товаров:')
print(test['item_nbr'].nunique())
print('\nПропуски:')
print(test.isna().sum())


Начальная дата test: 2017-08-16 00:00:00
Конечная дата test: 2017-08-31 00:00:00

Количество уникальных дат:
16

Количество уникальных магазинов:
54

Количество уникальных товаров:
3901

Пропуски:
id             0
date           0
store_nbr      0
item_nbr       0
onpromotion    0
dtype: int64


Тестовый период составляет 16 дней (с 16.08.2017 по 31.08.2017).

Для каждой требуемой пары (store_nbr, item_nbr) необходимо спрогнозировать unit_sales на следующие 16 дней.

Тогда validation тоже разумно сделать длиной 16 дней. Но, для начала, проверим, где заканчивается train.csv.

### 2.7 Обучающая выборка train.csv

Сначала загрузим 100 000 строк и проверим структуру:

In [150]:
train_sample = pd.read_csv(DATA_PATH / 'train.csv', nrows=100_000, parse_dates=['date'])
print('Размер sample:', train_sample.shape)
train_sample.head()


Размер sample: (100000, 6)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [151]:
train_sample.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id           100000 non-null  int64         
 1   date         100000 non-null  datetime64[ns]
 2   store_nbr    100000 non-null  int64         
 3   item_nbr     100000 non-null  int64         
 4   unit_sales   100000 non-null  float64       
 5   onpromotion  0 non-null       float64       
dtypes: datetime64[ns](1), float64(2), int64(3)
memory usage: 4.6 MB


In [152]:
train_path = DATA_PATH / 'train.csv'

with open(train_path, 'rb') as f:
    n_rows = sum(1 for _ in f) - 1  # -1, потому что первая строка - заголовок

print(f'Количество строк в train.csv: {n_rows:,}')

Количество строк в train.csv: 125,497,040


train.csv содержит больше 125 млн. строк, поэтому будем использовать чтение по частям.

In [153]:
chunk_size = 1000000

min_date = None
max_date = None

for chunk in pd.read_csv(
    train_path,
    usecols=['date'],
    chunksize=chunk_size,
    parse_dates=['date']
):
    chunk_min = chunk['date'].min()
    chunk_max = chunk['date'].max()

    if min_date is None or chunk_min < min_date:
        min_date = chunk_min

    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

print('Начальная дата train:', min_date)
print('Конечная дата train:', max_date)

Начальная дата train: 2013-01-01 00:00:00
Конечная дата train: 2017-08-15 00:00:00


Train заканчивается 15 августа 2017, а test начинается 16 августа 2017, никакого промежутка между ними нет.

Для валидации можем отрезать от конца train такой же горизонт - 16 дней.

Логика эксперимента будет такой: представляем, что сегодня 30 июля, всё после этой даты модель не видела, строим прогноз на следующие 16 дней, сравниваем его с реальными unit_sales.

Исследуем unit_sales на всём train: посмотрим, встречаются ли отрицательные продажи и насколько экстремальными бывают значения. Пройдём по файлу кусками и посчитаем основные характеристики.

In [154]:
chunk_size = 1000000

min_sales = np.inf
max_sales = -np.inf
negative_count = 0
zero_count = 0
total_count = 0
sales_sum = 0

for chunk in pd.read_csv(
    train_path,
    usecols=['unit_sales'],
    chunksize=chunk_size,
    dtype={'unit_sales': 'float32'}
):
    sales = chunk['unit_sales']

    min_sales = min(min_sales, sales.min())
    max_sales = max(max_sales, sales.max())

    negative_count += (sales < 0).sum()
    zero_count += (sales == 0).sum()

    total_count += len(sales)
    sales_sum += sales.sum()

mean_sales = sales_sum / total_count

print(f'Количество наблюдений: {total_count:,}')
print(f'Минимальное unit_sales: {min_sales:.3f}')
print(f'Максимальное unit_sales: {max_sales:.3f}')
print(f'Среднее unit_sales: {mean_sales:.3f}')

print(f'\nОтрицательных значений: {negative_count:,}')
print(f'Доля отрицательных: {negative_count / total_count * 100:.4f}%')

print(f'\nНулевых значений: {zero_count:,}')
print(f'Доля нулевых: {zero_count / total_count * 100:.4f}%')

Количество наблюдений: 125,497,040
Минимальное unit_sales: -15372.000
Максимальное unit_sales: 89440.000
Среднее unit_sales: 8.555

Отрицательных значений: 7,795
Доля отрицательных: 0.0062%

Нулевых значений: 0
Доля нулевых: 0.0000%


Всего в train 125497040 наблюдений, а среднее значение unit_sales около 8.56 единиц.
Диапазон значений большой - от -15372 до 89440. Вероятно, распределение продаж сильно скошено вправо.

Отрицательных наблюдений (возвратов) очень мало. Т.к. наша задача - прогнозировать спрос, позже заменим отрицательные значения на 0.

Теперь проверяем onpromotion на всём train:

In [155]:
promo_true = 0
promo_false = 0
promo_missing = 0
total_count = 0

for chunk in pd.read_csv(
    train_path,
    usecols=['onpromotion'],
    chunksize=1_000_000
):
    promo = chunk['onpromotion']

    promo_missing += promo.isna().sum()
    promo_true += (promo == True).sum()
    promo_false += (promo == False).sum()

    total_count += len(chunk)

print(f'Всего наблюдений: {total_count:,}')

print(f'\nPromotion = True: {promo_true:,}')
print(f'Доля: {promo_true / total_count * 100:.2f}%')

print(f'\nPromotion = False: {promo_false:,}')
print(f'Доля: {promo_false / total_count * 100:.2f}%')

print(f'\nPromotion = NaN: {promo_missing:,}')
print(f'Доля: {promo_missing / total_count * 100:.2f}%')

/tmp/ipykernel_58/2616739412.py:6: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



Всего наблюдений: 125,497,040

Promotion = True: 7,810,622
Доля: 6.22%

Promotion = False: 96,028,767
Доля: 76.52%

Promotion = NaN: 21,657,651
Доля: 17.26%


У 17.26% наблюдений информация об акции отсутствует. В используемом далее недавнем временном окне пропущенные значения onpromotion заполняются False. В test этот признак известен для всех наблюдений.

Теперь агрегируем продажи по дням во время чтения chunks. Получим ежедневные суммарные продажи:

In [156]:
daily_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=['date', 'unit_sales'],
    chunksize=1_000_000,
    dtype={'unit_sales': 'float32'}
):
    # Возвраты пока не должны уменьшать оценку спроса
    chunk['unit_sales_clean'] = chunk['unit_sales'].clip(lower=0)

    daily_chunk = (
        chunk
        .groupby('date', as_index=False)['unit_sales_clean']
        .sum()
    )

    daily_parts.append(daily_chunk)

daily_sales = (
    pd.concat(daily_parts)
    .groupby('date', as_index=False)['unit_sales_clean']
    .sum()
)

daily_sales['date'] = pd.to_datetime(daily_sales['date'])

daily_sales = daily_sales.sort_values('date')

print('Размер агрегированной таблицы:', daily_sales.shape)

daily_sales.head()

Размер агрегированной таблицы: (1684, 2)


,date,unit_sales_clean
0,2013-01-01,2511.618896
1,2013-01-02,496095.406250
2,2013-01-03,361487.312500
3,2013-01-04,354472.187500
4,2013-01-05,477357.125000


Получилось 1684 дня вместо 125,5 млн. строк.

На 1 января 2013 года: всего около 2,5 тыс. продаж, 2 января - почти 496 тыс. Это очень резкий контраст, нужно будет проверить его природу.

Построим динамику продаж:

In [157]:
fig = px.line(
    daily_sales,
    x='date',
    y='unit_sales_clean',
    title='Динамика суммарных ежедневных продаж Favorita'
)

fig.update_layout(
    xaxis_title='Дата',
    yaxis_title='Суммарные продажи',
    hovermode='x unified'
)

fig.show()


На графике виден восходящий тренд, регулярные колебания (возможно, недельная сезонность).

В начале каждого года наблюдаются провалы. Причину этих провалов по одному графику определить нельзя, поэтому дальше отдельно проверим календарные эффекты.

Также видно, что амплитуда колебаний со временем увеличивается.

Проверим недельную сезонность. Посчитаем средние продажи для каждого дня недели:

In [158]:
daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek

day_names = {
    0: 'Понедельник',
    1: 'Вторник',
    2: 'Среда',
    3: 'Четверг',
    4: 'Пятница',
    5: 'Суббота',
    6: 'Воскресенье'
}

daily_sales['day_name'] = (
    daily_sales['day_of_week']
    .map(day_names)
)

weekday_sales = (
    daily_sales
    .groupby(['day_of_week', 'day_name'], as_index=False)
    ['unit_sales_clean']
    .mean()
    .sort_values('day_of_week')
)

weekday_sales

,day_of_week,day_name,unit_sales_clean
0,0,Понедельник,617628.0625
1,1,Вторник,569985.1250
2,2,Среда,593298.8750
3,3,Четверг,505360.0625
4,4,Пятница,579638.3125
5,5,Суббота,772267.7500
6,6,Воскресенье,825255.0000


Максимальные средние продажи приходятся на воскресенье (около 825 тыс.), затем на субботу (около 772 тыс.), а минимальные - на четверг ( около 505 тыс.).

Визуализируем недельный паттерн:

In [159]:
fig = px.bar(
    weekday_sales,
    x='day_name',
    y='unit_sales_clean',
    title='Средние суммарные продажи по дням недели',
    text_auto='.0f'
)

fig.update_layout(
    xaxis_title='День недели',
    yaxis_title='Средние суммарные продажи'
)

fig.show()


В описании к соревнованию сказано, что землетрясение 16 апреля 2016 года и последовавшие сборы воды и товаров первой необходимости заметно повлияли на продажи в течение нескольких недель.

Посмотрим период вокруг землетрясения:

In [160]:
eq_date = pd.Timestamp('2016-04-16')

eq_period = daily_sales[
    (daily_sales['date'] >= '2016-03-15') &
    (daily_sales['date'] <= '2016-05-31')
].copy()

fig = px.line(
    eq_period,
    x='date',
    y='unit_sales_clean',
    title='Продажи Favorita до и после землетрясения 16 апреля 2016'
)

fig.add_vline(
    x=eq_date.timestamp() * 1000,
    line_dash='dash',
    annotation_text='Землетрясение 16.04.2016',
    annotation_position='top'
)

fig.update_layout(
    xaxis_title='Дата',
    yaxis_title='Суммарные продажи',
    hovermode='x unified'
)

fig.show()


До 16 апреля присутствуют обычные недельные колебания. Сразу после даты землетрясения продажи резко поднимаются: примерно до 1.27 млн 17 апреля и 1.34 млн 18 апреля. Затем повышенный уровень сохраняется несколько дней.

Но и до землетрясения встречались большие пики. Например, в начале апреля около 1.26 млн. Поэтому одного графика недостаточно, чтобы утверждать, что «землетрясение увеличило продажи». Нужно отделить этот эффект от обычной недельной сезонности.

Сравним 14 дней "до" и 14 дней "после":

In [161]:
before_eq = daily_sales[
    (daily_sales['date'] >= '2016-04-02') &
    (daily_sales['date'] <= '2016-04-15')
]['unit_sales_clean']

after_eq = daily_sales[
    (daily_sales['date'] >= '2016-04-17') &
    (daily_sales['date'] <= '2016-04-30')
]['unit_sales_clean']

before_mean = before_eq.mean()
after_mean = after_eq.mean()

change_pct = (after_mean / before_mean - 1) * 100

print(f'Средние продажи за 14 дней ДО:    {before_mean:,.0f}')
print(f'Средние продажи за 14 дней ПОСЛЕ: {after_mean:,.0f}')
print(f'Изменение: {change_pct:+.2f}%')


Средние продажи за 14 дней ДО:    793,035
Средние продажи за 14 дней ПОСЛЕ: 938,414
Изменение: +18.33%


Cредние суммарные продажи в течение двух недель после землетрясения наблюдались на 18.33% выше, чем две недели до него.

Проверим пропущенные даты.

У нас получилось 1684 строки daily_sales. Период с 01.01.2013 по 15.08.2017 содержит больше календарных дней. Значит, нужно проверить: есть ли даты, полностью отсутствующие в train.

In [162]:
full_calendar = pd.date_range(
    start=daily_sales['date'].min(),
    end=daily_sales['date'].max(),
    freq='D'
)

missing_dates = full_calendar.difference(daily_sales['date'])

print(f'Всего календарных дней: {len(full_calendar)}')
print(f'Дней в train: {len(daily_sales)}')
print(f'Полностью отсутствующих дат: {len(missing_dates)}')

missing_dates


Всего календарных дней: 1688
Дней в train: 1684
Полностью отсутствующих дат: 4


DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]', freq=None)

В train полностью отсутствуют четыре даты: 25 декабря каждого года.

Для дальнейшего анализа восстановим непрерывный календарь, добавив пропущенные даты, но оставим продажи NaN:

In [163]:
daily_sales_full = (
    daily_sales[['date', 'unit_sales_clean']]
    .set_index('date')
    .reindex(full_calendar)
    .rename_axis('date')
    .reset_index()
)

print('Размер:', daily_sales_full.shape)
print('\nКоличество NaN:')
print(daily_sales_full.isna().sum())

print('\nПропущенные даты:')
display(
    daily_sales_full[
        daily_sales_full['unit_sales_clean'].isna()
    ]
)

Размер: (1688, 2)

Количество NaN:
date                0
unit_sales_clean    4
dtype: int64

Пропущенные даты:


,date,unit_sales_clean
358,2013-12-25,NaN
723,2014-12-25,NaN
1088,2015-12-25,NaN
1454,2016-12-25,NaN


## 3. Схема валидации

In [164]:
VAL_START = pd.Timestamp('2017-07-31')
VAL_END = pd.Timestamp('2017-08-15')

TRAIN_END = VAL_START - pd.Timedelta(days=1)

FORECAST_HORIZON = 16

print('Конец обучения:', TRAIN_END.date())
print('Начало validation:', VAL_START.date())
print('Конец validation:', VAL_END.date())

print(
    'Горизонт validation:',
    (VAL_END - VAL_START).days + 1,
    'дней'
)

print(
    'Горизонт Kaggle test:',
    test['date'].nunique(),
    'дней'
)

Конец обучения: 2017-07-30
Начало validation: 2017-07-31
Конец validation: 2017-08-15
Горизонт validation: 16 дней
Горизонт Kaggle test: 16 дней


## 4. Метрики качества

Для оценки качества моделей используем две метрики:
- NWRMSLE - основная метрика, соответствующая метрике соревнования. Ошибка рассчитывается в логарифмической шкале, а для скоропортящихся товаров используется вес 1.25 вместо 1.0.
- MAE - дополнительная метрика на исходной шкале продаж. Она показывает среднюю абсолютную ошибку прогноза в единицах товара.

Для моделей, обучаемых на log1p(target), прогноз перед расчётом MAE преобразуется обратно в исходную шкалу с помощью expm1. Отрицательные прогнозы ограничиваются нулём.

Создадим функции метрик:

In [165]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return np.mean(np.abs(y_true - y_pred))


def nwrmsle(y_true, y_pred, weights=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Для логарифма прогнозы должны быть неотрицательными
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)

    if weights is None:
        weights = np.ones(len(y_true))
    else:
        weights = np.asarray(weights)

    squared_log_error = (
        np.log1p(y_pred) - np.log1p(y_true)
    ) ** 2

    return np.sqrt(
        np.sum(weights * squared_log_error) /
        np.sum(weights)
    )


## 5. Baseline-модели

### 5.1 Naive

Читаем только validation из train.csv:

In [166]:
val_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=[
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales'
    ],
    chunksize=1000000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date']
):
    val_chunk = chunk[
        (chunk['date'] >= VAL_START) &
        (chunk['date'] <= VAL_END)
    ].copy()

    if len(val_chunk) > 0:
        val_parts.append(val_chunk)

validation = pd.concat(
    val_parts,
    ignore_index=True
)

# Возвраты для нашей целевой переменной считаем нулевым спросом
validation['unit_sales_clean'] = (
    validation['unit_sales']
    .clip(lower=0)
)

print('Размер validation:', validation.shape)
print(
    'Период:',
    validation['date'].min().date(),
    '—',
    validation['date'].max().date()
)
print(
    'Количество дней:',
    validation['date'].nunique()
)
print(
    'Магазинов:',
    validation['store_nbr'].nunique()
)
print(
    'Товаров:',
    validation['item_nbr'].nunique()
)

validation.head()


Размер validation: (1677344, 5)
Период: 2017-07-31 — 2017-08-15
Количество дней: 16
Магазинов: 54
Товаров: 3857


,date,store_nbr,item_nbr,unit_sales,unit_sales_clean
0,2017-07-31,1,96995,2.0,2.0
1,2017-07-31,1,103520,1.0,1.0
2,2017-07-31,1,103665,2.0,2.0
3,2017-07-31,1,105574,4.0,4.0
4,2017-07-31,1,105575,12.0,12.0


Нужно найти последнюю продажу, известную к моменту начала validation, то есть не позднее 2017-07-30.

Если просто взять последнюю строку для товара за всю историю, у редко продающегося товара это значение может оказаться, например, месячной давности.

Формально это всё равно Naive по последнему наблюдению, но нужно проверить, насколько старые значения используются.

In [167]:
last_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=[
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales'
    ],
    chunksize=1000000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date']
):
    # Используем только прошлое: никаких данных из validation
    chunk = chunk[
        chunk['date'] <= TRAIN_END
    ].copy()

    if len(chunk) == 0:
        continue

    # Последнее наблюдение каждой пары внутри текущего chunk
    chunk_last = (
        chunk
        .sort_values('date')
        .groupby(
            ['store_nbr', 'item_nbr'],
            as_index=False
        )
        .tail(1)
    )

    last_parts.append(chunk_last)

last_sales = pd.concat(
    last_parts,
    ignore_index=True
)

# Одна и та же пара могла встретиться в разных chunks, поэтому ещё раз оставляем самое позднее наблюдение
last_sales = (
    last_sales
    .sort_values('date')
    .groupby(
        ['store_nbr', 'item_nbr'],
        as_index=False
    )
    .tail(1)
    .reset_index(drop=True)
)

last_sales['unit_sales_clean'] = (
    last_sales['unit_sales']
    .clip(lower=0)
)

print('Количество пар store-item:', len(last_sales))

print(
    'Самая ранняя "последняя продажа":',
    last_sales['date'].min().date()
)

print(
    'Самая поздняя "последняя продажа":',
    last_sales['date'].max().date()
)

last_sales.head()


Количество пар store-item: 174140
Самая ранняя "последняя продажа": 2013-01-02
Самая поздняя "последняя продажа": 2017-07-30


,date,store_nbr,item_nbr,unit_sales,unit_sales_clean
0,2013-01-02,2,699688,25.138,25.138
1,2013-01-02,43,892076,1.000,1.000
2,2013-01-02,34,892076,1.000,1.000
3,2013-01-03,1,699688,10.406,10.406
4,2013-01-03,39,802833,3.000,3.000


Самое позднее использованное наблюдение 2017-07-30. Значит, утечки из validation нет. Самая ранняя «последняя продажа» 2013-01-02. То есть есть пары, которые продавались в 2013 году, а потом больше вообще не встречались. Если такую пару встретить в validation, странно прогнозировать её продажи значением четырёхлетней давности.

Проверим покрытие validation историей.

Найдём уникальные пары validation, а потом сопоставим их с last_sales:

In [168]:
val_pairs = (
    validation[
        ['store_nbr', 'item_nbr']
    ]
    .drop_duplicates()
)

val_pairs_check = val_pairs.merge(
    last_sales[
        ['store_nbr', 'item_nbr', 'date', 'unit_sales_clean']
    ],
    on=['store_nbr', 'item_nbr'],
    how='left'
)

known_pairs = val_pairs_check['unit_sales_clean'].notna().sum()
unknown_pairs = val_pairs_check['unit_sales_clean'].isna().sum()
total_pairs = len(val_pairs_check)

print(f'Уникальных пар в validation: {total_pairs:,}')

print(
    f'Есть история до validation: {known_pairs:,} '
    f'({known_pairs / total_pairs * 100:.2f}%)'
)

print(
    f'Нет истории до validation: {unknown_pairs:,} '
    f'({unknown_pairs / total_pairs * 100:.2f}%)'
)


Уникальных пар в validation: 146,648
Есть история до validation: 146,103 (99.63%)
Нет истории до validation: 545 (0.37%)


Посчитаем, сколько дней прошло между последним известным наблюдением пары и концом обучающего периода 2017-07-30:

In [169]:
val_pairs_check['days_since_last_sale'] = (
    TRAIN_END - val_pairs_check['date']
).dt.days

known_history = val_pairs_check[
    val_pairs_check['days_since_last_sale'].notna()
]

print('Возраст последнего наблюдения (дни):')
print(
    known_history['days_since_last_sale']
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print('\nДоля пар, последнее наблюдение которых было:')

for days in [0, 1, 7, 14, 30, 90]:
    share = (
        known_history['days_since_last_sale'] <= days
    ).mean() * 100

    print(f'не более {days:>2} дней назад: {share:.2f}%')


Возраст последнего наблюдения (дни):
count    146103.000000
mean          2.349651
std          21.486698
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
90%           3.000000
95%           7.000000
99%          37.000000
max        1275.000000
Name: days_since_last_sale, dtype: float64

Доля пар, последнее наблюдение которых было:
не более  0 дней назад: 75.91%
не более  1 дней назад: 86.43%
не более  7 дней назад: 95.64%
не более 14 дней назад: 97.61%
не более 30 дней назад: 98.83%
не более 90 дней назад: 99.55%


У 75.91% пар последнее наблюдение приходится на 2017-07-30, у 95.64% оно не старше недели. Медиана равна 0 дней. Есть редкие экстремальные случаи: максимум - 1275 дней, но это небольшой хвост.

Простой Naive имеет смысл как baseline, хотя для редких товаров он может быть слабым.

Если мы оценим Naive только на существующих строках validation, то фактически проигнорируем случаи, когда товар в конкретном магазине в конкретный день не продавался. Метрика получится несопоставимой с реальной задачей.

Для оценки baseline построим полную 16-дневную validation-панель. В неё включим пары, активные непосредственно перед началом validation, а также пары, которые встречаются в самом validation-периоде. Для каждой пары создадим все 16 календарных дат, а отсутствующие записи продаж будем считать нулевыми.

Проверим структуру самого test: одинаковый ли набор пар присутствует во все 16 дней:

In [170]:
test_pairs_by_date = (
    test
    .groupby('date')
    .size()
    .reset_index(name='n_rows')
)

print(test_pairs_by_date)

print(
    '\nОдинаковое число строк каждый день:',
    test_pairs_by_date['n_rows'].nunique() == 1
)

test_unique_pairs = (
    test[['store_nbr', 'item_nbr']]
    .drop_duplicates()
)

print(
    '\nУникальных пар store-item в test:',
    len(test_unique_pairs)
)

print(
    'Число пар × 16 дней:',
    len(test_unique_pairs) * 16
)

print(
    'Всего строк test:',
    len(test)
)


         date  n_rows
0  2017-08-16  210654
1  2017-08-17  210654
2  2017-08-18  210654
3  2017-08-19  210654
4  2017-08-20  210654
5  2017-08-21  210654
6  2017-08-22  210654
7  2017-08-23  210654
8  2017-08-24  210654
9  2017-08-25  210654
10 2017-08-26  210654
11 2017-08-27  210654
12 2017-08-28  210654
13 2017-08-29  210654
14 2017-08-30  210654
15 2017-08-31  210654

Одинаковое число строк каждый день: True

Уникальных пар store-item в test: 210654
Число пар × 16 дней: 3370464
Всего строк test: 3370464


Во все 16 дней test используется один и тот же фиксированный набор из 210 654 пар (store, item)

Посмотрим, сколько уникальных (store, item) встречалось непосредственно перед 2017-07-31.

Возьмём последние 16 дней training - симметрично горизонту прогноза:

In [171]:
ACTIVE_START = TRAIN_END - pd.Timedelta(days=15)

active_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=['date', 'store_nbr', 'item_nbr'],
    chunksize=1_000_000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32'
    },
    parse_dates=['date']
):
    active_chunk = chunk[
        (chunk['date'] >= ACTIVE_START) &
        (chunk['date'] <= TRAIN_END)
    ][['store_nbr', 'item_nbr']]

    if len(active_chunk) > 0:
        active_parts.append(
            active_chunk.drop_duplicates()
        )

active_pairs = (
    pd.concat(active_parts, ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    'Период определения активных пар:',
    ACTIVE_START.date(),
    '—',
    TRAIN_END.date()
)

print(
    'Активных пар store-item:',
    f'{len(active_pairs):,}'
)

print(
    'Уникальных пар непосредственно в validation:',
    f'{len(val_pairs):,}'
)


Период определения активных пар: 2017-07-15 — 2017-07-30
Активных пар store-item: 147,381
Уникальных пар непосредственно в validation: 146,648


Количества очень близкие. Но прежде чем строить панель, нужно проверить, сколько активных пар вообще не продавались в validation и сколько, наоборот, появилось в validation, хотя не было активно в предыдущие 16 дней. Это покажет, насколько разумно наше определение «активной серии»:

In [172]:
pair_comparison = active_pairs.merge(
    val_pairs,
    on=['store_nbr', 'item_nbr'],
    how='outer',
    indicator=True
)

comparison_counts = pair_comparison['_merge'].value_counts()

both = comparison_counts.get('both', 0)
only_active = comparison_counts.get('left_only', 0)
only_validation = comparison_counts.get('right_only', 0)

print(f'Есть и до validation, и в validation: {both:,}')
print(f'Активны до validation, но не встретились в validation: {only_active:,}')
print(f'Появились в validation, но не были активны до него: {only_validation:,}')

print(
    '\nДоля новых/вернувшихся пар в validation:',
    f'{only_validation / len(val_pairs) * 100:.2f}%'
)


Есть и до validation, и в validation: 142,869
Активны до validation, но не встретились в validation: 4,512
Появились в validation, но не были активны до него: 3,779

Доля новых/вернувшихся пар в validation: 2.58%


Если построить validation только из active_pairs, мы потеряем 3 779 серий.

Объединим два набора: активные до validation и встретившиеся в validation. Для каждой пары создадим 16 календарных дней.

Если для некоторой пары в какой-то день записи в исходном train нет, в конкурсной постановке для нашей validation-панели будем трактовать её как unit_sales = 0.

Так мы одновременно сохраним серии, которые были активны на момент прогноза, и не выбросим появившиеся в validation серии.

In [173]:
# Объединяем пары
validation_pairs = (
    pd.concat(
        [
            active_pairs[['store_nbr', 'item_nbr']],
            val_pairs[['store_nbr', 'item_nbr']]
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

# Все 16 дат validation
validation_dates = pd.DataFrame({
    'date': pd.date_range(
        VAL_START,
        VAL_END,
        freq='D'
    )
})

# Декартово произведение: каждая пара x каждый день
validation_pairs['_key'] = 1
validation_dates['_key'] = 1

validation_full = (
    validation_pairs
    .merge(validation_dates, on='_key')
    .drop(columns='_key')
)

# Добавляем реальные продажи
validation_full = validation_full.merge(
    validation[
        ['date', 'store_nbr', 'item_nbr', 'unit_sales_clean']
    ],
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# Отсутствующая запись = 0 продаж
validation_full['unit_sales_clean'] = (
    validation_full['unit_sales_clean']
    .fillna(0)
    .astype('float32')
)

print(
    'Уникальных пар:',
    f'{len(validation_pairs):,}'
)

print(
    'Строк полной validation:',
    f'{len(validation_full):,}'
)

print(
    'Ожидаем:',
    f'{len(validation_pairs) * FORECAST_HORIZON:,}'
)

print(
    'NaN в target:',
    validation_full['unit_sales_clean'].isna().sum()
)

validation_full.head()


Уникальных пар: 151,160
Строк полной validation: 2,418,560
Ожидаем: 2,418,560
NaN в target: 0


,store_nbr,item_nbr,date,unit_sales_clean
0,1,99197,2017-07-31,0.0
1,1,99197,2017-08-01,0.0
2,1,99197,2017-08-02,0.0
3,1,99197,2017-08-03,0.0
4,1,99197,2017-08-04,0.0


Построим Naive-прогноз.

last_sales содержит последнее известное наблюдение каждой пары не позднее 30 июля 2017.

Для пар, у которых вообще нет истории, Naive не может получить последнее значение. В качестве fallback дадим им прогноз 0.

In [174]:
# Оставляем нужные столбцы и переименовываем прогноз
naive_values = (
    last_sales[
        ['store_nbr', 'item_nbr', 'unit_sales_clean']
    ]
    .rename(
        columns={'unit_sales_clean': 'naive_pred'}
    )
)

# Добавляем последнее известное значение к полной validation
validation_naive = validation_full.merge(
    naive_values,
    on=['store_nbr', 'item_nbr'],
    how='left'
)

# Если история пары отсутствует - fallback = 0
missing_naive = validation_naive['naive_pred'].isna().sum()

validation_naive['naive_pred'] = (
    validation_naive['naive_pred']
    .fillna(0)
    .astype('float32')
)

print(
    'Строк без исторического значения до fallback:',
    f'{missing_naive:,}'
)

print(
    'Доля таких строк:',
    f'{missing_naive / len(validation_naive) * 100:.4f}%'
)

validation_naive[
    [
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales_clean',
        'naive_pred'
    ]
].head(10)


Строк без исторического значения до fallback: 8,720
Доля таких строк: 0.3605%


,date,store_nbr,item_nbr,unit_sales_clean,naive_pred
0,2017-07-31,1,99197,0.0,1.0
1,2017-08-01,1,99197,0.0,1.0
2,2017-08-02,1,99197,0.0,1.0
3,2017-08-03,1,99197,0.0,1.0
4,2017-08-04,1,99197,0.0,1.0
5,2017-08-05,1,99197,1.0,1.0
6,2017-08-06,1,99197,0.0,1.0
7,2017-08-07,1,99197,2.0,1.0
8,2017-08-08,1,99197,0.0,1.0
9,2017-08-09,1,99197,2.0,1.0


Получили 8720 строк без исторического значения, то есть всего 0.3605% полной validation-панели.

 Поскольку каждая пара повторяется 16 раз, это соответствует: 8720 / 16 = 545 парам - ровно тем 545 cold-start парам, которые мы обнаружили раньше. Значит, все предыдущие расчёты согласуются между собой.

 Первые строки хорошо показывают слабость Naive. Для (store=1, item=99197) последнее известное значение было 1, поэтому Naive прогнозирует 1 каждый день. В действительности продажи выглядят как 0, 0, 0, 0, 0, 1, 0, 2, 0, 2.... То есть для разреженных продаж постоянный прогноз может систематически давать продажи в дни, когда фактически их нет. Позже SeasonalNaive и модели с лагами/днём недели должны с этим работать лучше.

 Добавляем веса perishable:

In [175]:
validation_naive = validation_naive.merge(
    items[['item_nbr', 'perishable']],
    on='item_nbr',
    how='left'
)

# Проверяем, все ли товары нашли в items.csv
print(
    'Пропусков в perishable:',
    validation_naive['perishable'].isna().sum()
)

# Создаём веса
validation_naive['weight'] = np.where(
    validation_naive['perishable'] == 1,
    1.25,
    1.0
).astype('float32')

print('\nРаспределение perishable:')
print(
    validation_naive['perishable']
    .value_counts()
    .sort_index()
)

print('\nРаспределение весов:')
print(
    validation_naive['weight']
    .value_counts()
    .sort_index()
)

Пропусков в perishable: 0

Распределение perishable:
perishable
0    1882688
1     535872
Name: count, dtype: int64

Распределение весов:
weight
1.00    1882688
1.25     535872
Name: count, dtype: int64


Всё сошлось. Пропусков в perishable 0, значит каждый товар корректно сопоставился со справочником. Обычных наблюдений 1882688, вес 1. Скоропортящихся 535872, вес 1.25. Сумма равна нашим 2418560 строкам validation.

Можем переходить к расчету метрик.

In [176]:
naive_mae = mae(
    validation_naive['unit_sales_clean'],
    validation_naive['naive_pred']
)

naive_nwrmsle = nwrmsle(
    validation_naive['unit_sales_clean'],
    validation_naive['naive_pred'],
    weights=validation_naive['weight']
)

print(f'Naive MAE:      {naive_mae:.4f}')
print(f'Naive NWRMSLE:  {naive_nwrmsle:.4f}')

Naive MAE:      5.2521
Naive NWRMSLE:  0.9414


Простой Naive в среднем ошибается примерно на 5.25 единицы товара на строку (дата, магазин, товар). А NWRMSLE = 0.9414 теперь будет ориентиром: следующие модели должны стремиться дать значение ниже.

Naive выдаёт одно и то же последнее известное значение на все 16 дней. Но мы уже заметили зависимость продаж от дня недели. Поэтому логичный следующий кандидат SeasonalNaive.

### 5.2 SeasonalNaive (период 7 дней)

Понедельник прогнозируем предыдущим понедельником, вторник - предыдущим вторником и т.д.

Но validation длится 16 дней, поэтому появляется проблема: для 31 июля значение t-7 - 24 июля, оно находится в training. А для 7 августа значение t-7 - 31 июля, это уже validation.

Мы не имеем права использовать фактические продажи 31 июля для прогнозирования 7 августа, если имитируем ситуацию, в которой 30 июля сразу строим прогноз на все следующие 16 дней. Это была бы утечка данных.

SeasonalNaive должен быть рекурсивным: для второй и третьей недели повторяем прогноз первой недели, а не подглядываем в validation.

Сначала получим последние 7 календарных дней training:

In [177]:
SEASONAL_START = TRAIN_END - pd.Timedelta(days=6)

seasonal_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=[
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales'
    ],
    chunksize=1_000_000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date']
):
    seasonal_chunk = chunk[
        (chunk['date'] >= SEASONAL_START) &
        (chunk['date'] <= TRAIN_END)
    ].copy()

    if len(seasonal_chunk) > 0:
        seasonal_parts.append(seasonal_chunk)

seasonal_history = pd.concat(
    seasonal_parts,
    ignore_index=True
)

seasonal_history['unit_sales_clean'] = (
    seasonal_history['unit_sales']
    .clip(lower=0)
)

print(
    'Период сезонной истории:',
    seasonal_history['date'].min().date(),
    '—',
    seasonal_history['date'].max().date()
)

print(
    'Количество дней:',
    seasonal_history['date'].nunique()
)

print(
    'Количество строк:',
    f'{len(seasonal_history):,}'
)

print(
    'Уникальных пар:',
    f"{seasonal_history[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]:,}"
)


Период сезонной истории: 2017-07-24 — 2017-07-30
Количество дней: 7
Количество строк: 729,897
Уникальных пар: 141,098


Строк получилось меньше 151160 х 7 = 1058120, потому что исходный train разреженный.

Перед SeasonalNaive нам нужно восстановить полную сетку этих семи дней для всех 151160 validation-пар. Отсутствующие записи внутри этой сетки будем считать нулевыми продажами - так же, как мы сделали при формировании validation.

Создаём 7-дневный сезонный шаблон:

In [178]:
# Все 7 календарных дат перед validation
seasonal_dates = pd.DataFrame({
    'date': pd.date_range(
        SEASONAL_START,
        TRAIN_END,
        freq='D'
    )
})

# Берём те же пары, на которых оцениваем validation
seasonal_pairs = validation_pairs[
    ['store_nbr', 'item_nbr']
].copy()

# Создаём полную сетку: каждая пара × каждый из 7 дней
seasonal_pairs['_key'] = 1
seasonal_dates['_key'] = 1

seasonal_full = (
    seasonal_pairs
    .merge(seasonal_dates, on='_key')
    .drop(columns='_key')
)

# Добавляем реальные продажи за 24–30 июля
seasonal_full = seasonal_full.merge(
    seasonal_history[
        ['date', 'store_nbr', 'item_nbr', 'unit_sales_clean']
    ],
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# Если строки в train не было, считаем продажи равными 0
seasonal_full['unit_sales_clean'] = (
    seasonal_full['unit_sales_clean']
    .fillna(0)
    .astype('float32')
)

print(
    'Размер сезонной панели:',
    f'{len(seasonal_full):,}'
)

print(
    'Ожидаемый размер:',
    f'{len(validation_pairs) * 7:,}'
)

print(
    'NaN после заполнения:',
    seasonal_full['unit_sales_clean'].isna().sum()
)

print('\nКоличество строк по датам:')
print(
    seasonal_full
    .groupby('date')
    .size()
)


Размер сезонной панели: 1,058,120
Ожидаемый размер: 1,058,120
NaN после заполнения: 0

Количество строк по датам:
date
2017-07-24    151160
2017-07-25    151160
2017-07-26    151160
2017-07-27    151160
2017-07-28    151160
2017-07-29    151160
2017-07-30    151160
dtype: int64


Теперь пропусков нет. Можем получить второй baseline и сравнить его с Naive.

Строим SeasonalNaive и считаем метрики:

In [179]:
# Номер дня недели для 7-дневного шаблона
seasonal_full['day_of_week'] = (
    seasonal_full['date'].dt.dayofweek
)

seasonal_template = (
    seasonal_full[
        [
            'store_nbr',
            'item_nbr',
            'day_of_week',
            'unit_sales_clean'
        ]
    ]
    .rename(
        columns={
            'unit_sales_clean': 'seasonal_naive_pred'
        }
    )
)

# Копируем validation
validation_seasonal = validation_full.copy()

# Определяем день недели каждой строки validation
validation_seasonal['day_of_week'] = (
    validation_seasonal['date'].dt.dayofweek
)

# Присоединяем значение соответствующего дня из недели 24–30 июля
validation_seasonal = validation_seasonal.merge(
    seasonal_template,
    on=[
        'store_nbr',
        'item_nbr',
        'day_of_week'
    ],
    how='left'
)

print(
    'NaN в SeasonalNaive:',
    validation_seasonal['seasonal_naive_pred']
    .isna()
    .sum()
)

# Добавляем веса товаров
validation_seasonal = validation_seasonal.merge(
    items[['item_nbr', 'perishable']],
    on='item_nbr',
    how='left'
)

validation_seasonal['weight'] = np.where(
    validation_seasonal['perishable'] == 1,
    1.25,
    1.0
).astype('float32')

# Считаем метрики
seasonal_mae = mae(
    validation_seasonal['unit_sales_clean'],
    validation_seasonal['seasonal_naive_pred']
)

seasonal_nwrmsle = nwrmsle(
    validation_seasonal['unit_sales_clean'],
    validation_seasonal['seasonal_naive_pred'],
    weights=validation_seasonal['weight']
)

print(f'\nSeasonalNaive MAE:     {seasonal_mae:.4f}')
print(f'SeasonalNaive NWRMSLE: {seasonal_nwrmsle:.4f}')

print(f'\nNaive MAE:             {naive_mae:.4f}')
print(f'Naive NWRMSLE:         {naive_nwrmsle:.4f}')


NaN в SeasonalNaive: 0

SeasonalNaive MAE:     3.9672
SeasonalNaive NWRMSLE: 0.8794

Naive MAE:             5.2521
Naive NWRMSLE:         0.9414


Видим реальное улучшение baseline по обеим метрикам.

MAE снизился примерно на 24.46%, а NWRMSLE - примерно на 6.59%. То есть учёт недельной сезонности действительно помогает. Это хорошо согласуется с нашим EDA, где суббота и воскресенье заметно отличались от будних дней.

Теперь посмотрим, насколько разрежены наши ряды:

In [180]:
zero_share = (
    validation_full['unit_sales_clean'].eq(0).mean()
)

positive_share = (
    validation_full['unit_sales_clean'].gt(0).mean()
)

print(
    f'Доля нулевых наблюдений: '
    f'{zero_share * 100:.2f}%'
)

print(
    f'Доля положительных наблюдений: '
    f'{positive_share * 100:.2f}%'
)

# Для каждой серии считаем, в скольких из 16 дней были положительные продажи
series_activity = (
    validation_full
    .groupby(['store_nbr', 'item_nbr'])
    ['unit_sales_clean']
    .apply(lambda x: (x > 0).sum())
)

print('\nЧисло дней с продажами из 16:')
print(
    series_activity.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


Доля нулевых наблюдений: 30.65%
Доля положительных наблюдений: 69.35%

Число дней с продажами из 16:
count    151160.000000
mean         11.095654
std           4.901028
min           0.000000
25%           8.000000
50%          13.000000
75%          15.000000
90%          16.000000
95%          16.000000
99%          16.000000
max          16.000000
Name: unit_sales_clean, dtype: float64


В полной validation-панели 69.35% наблюдений положительные, а нулевых - 30.65%. Медианная серия продаётся в 13 из 16 дней, 75% серий - как минимум в 8 из 16 дней, а как минимум 10% серий имеют продажи вообще во все 16 дней.

Проблема применимости ETS и Theta сейчас заключается в масштабе - около 151 тыс. отдельных рядов и более четырёх лет ежедневной истории. Установим библиотеку StatsForecast для эффективной работы с большим количеством рядов. В ней есть быстрые реализации AutoETS и AutoTheta.

In [181]:
!pip install statsforecast -q


In [182]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoTheta


Возьмём последние 90 дней до validation. Это даст каждой модели достаточно истории, чтобы увидеть недельную динамику и локальный уровень, но сохранит вычисления в разумных пределах.

In [183]:
CLASSICAL_HISTORY_DAYS = 90

CLASSICAL_START = (
    TRAIN_END
    - pd.Timedelta(days=CLASSICAL_HISTORY_DAYS - 1)
)

print(
    'История для ETS/Theta:',
    CLASSICAL_START.date(),
    '—',
    TRAIN_END.date()
)

print(
    'Количество календарных дней:',
    (TRAIN_END - CLASSICAL_START).days + 1
)

История для ETS/Theta: 2017-05-02 — 2017-07-30
Количество календарных дней: 90


Прочитаем из огромного train.csv только этот период:

In [184]:
classical_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=[
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales'
    ],
    chunksize=1_000_000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date']
):
    classical_chunk = chunk[
        (chunk['date'] >= CLASSICAL_START) &
        (chunk['date'] <= TRAIN_END)
    ].copy()

    if len(classical_chunk) > 0:
        classical_parts.append(classical_chunk)

classical_history = pd.concat(
    classical_parts,
    ignore_index=True
)

classical_history['unit_sales_clean'] = (
    classical_history['unit_sales']
    .clip(lower=0)
    .astype('float32')
)

print(
    'Период:',
    classical_history['date'].min().date(),
    '—',
    classical_history['date'].max().date()
)

print(
    'Количество дней:',
    classical_history['date'].nunique()
)

print(
    'Количество строк:',
    f'{len(classical_history):,}'
)

print(
    'Уникальных пар:',
    f"{classical_history[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]:,}"
)

Период: 2017-05-02 — 2017-07-30
Количество дней: 90
Количество строк: 9,526,330
Уникальных пар: 158,530


За 90 дней встречалось 158530 пар, а наша validation-панель содержит 151160 пар. Некоторые товары были активны раньше, но к validation уже перестали быть релевантными.

Не будем строить полную панель для всех 158530 пар. Для оценки ETS/Theta нас интересуют те же 151160 серий, на которых мы сравнивали Naive и SeasonalNaive. Иначе сравнение моделей получится нечестным.

Полная 90-дневная панель для них будет иметь: 151160 х 90 = 13604400 строк. Это уже существенно больше, чем предыдущие таблицы, но гораздо меньше 255 млн.

Перед её созданием проверим, сколько из validation-пар встречались за эти 90 дней:

In [185]:
classical_pairs = (
    classical_history[
        ['store_nbr', 'item_nbr']
    ]
    .drop_duplicates()
)

classical_coverage = validation_pairs.merge(
    classical_pairs.assign(has_90d_history=1),
    on=['store_nbr', 'item_nbr'],
    how='left'
)

n_with_history = (
    classical_coverage['has_90d_history']
    .notna()
    .sum()
)

n_without_history = (
    classical_coverage['has_90d_history']
    .isna()
    .sum()
)

print(
    'Validation-пар всего:',
    f'{len(validation_pairs):,}'
)

print(
    'Есть история за последние 90 дней:',
    f'{n_with_history:,}',
    f'({n_with_history / len(validation_pairs) * 100:.2f}%)'
)

print(
    'Нет истории за последние 90 дней:',
    f'{n_without_history:,}',
    f'({n_without_history / len(validation_pairs) * 100:.2f}%)'
)

print(
    '\nРазмер полной панели 90 дней:',
    f'{len(validation_pairs) * CLASSICAL_HISTORY_DAYS:,}',
    'строк'
)

Validation-пар всего: 151,160
Есть история за последние 90 дней: 149,950 (99.20%)
Нет истории за последние 90 дней: 1,210 (0.80%)

Размер полной панели 90 дней: 13,604,400 строк


99.2% validation-серий имеют историю в выбранном 90-дневном окне, а без неё остаются только 1210 пар (0.8%). Значит, 90 дней как вычислительный компромисс выглядит приемлемо.

Для AutoETS и AutoTheta построение моделей отдельно для всех 150 тыс. временных рядов требует значительных вычислительных ресурсов. Можем оценить эти модели на фиксированной выборке из 5000 рядов.

Для воспроизводимости зададим random_state = 42.

Метрики для этих моделей считаются только на выбранных рядах и поэтому не сравниваются напрямую с результатами Naive и SeasonalNaive на полной validation. Для корректного сравнения дополнительно пересчитаем Naive и SeasonalNaive на тех же 5000 рядах.

Выбираем 5000 рядов:

In [186]:
CLASSICAL_SAMPLE_SIZE = 5000

classical_sample_pairs = (
    classical_coverage[
        classical_coverage['has_90d_history'].notna()
    ][['store_nbr', 'item_nbr']]
    .sample(
        n=CLASSICAL_SAMPLE_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

print(
    'Количество выбранных рядов:',
    f'{len(classical_sample_pairs):,}'
)

print(
    'Будет строк истории:',
    f'{len(classical_sample_pairs) * CLASSICAL_HISTORY_DAYS:,}'
)

print(
    'Будет validation-наблюдений:',
    f'{len(classical_sample_pairs) * FORECAST_HORIZON:,}'
)

classical_sample_pairs.head()


Количество выбранных рядов: 5,000
Будет строк истории: 450,000
Будет validation-наблюдений: 80,000


,store_nbr,item_nbr
0,41,1091366
1,4,2060787
2,28,1457240
3,43,1973587
4,37,1964850


Формируем 90-дневную панель для 5000 рядов:

In [187]:
# Оставляем историю только выбранных 5000 рядов
classical_sample_history = classical_history.merge(
    classical_sample_pairs,
    on=['store_nbr', 'item_nbr'],
    how='inner'
)

# Полный календарь из 90 дней
classical_dates = pd.DataFrame({
    'date': pd.date_range(
        CLASSICAL_START,
        TRAIN_END,
        freq='D'
    )
})

# Создаём полную сетку: 5000 рядов × 90 дней
classical_grid = (
    classical_sample_pairs
    .assign(_key=1)
    .merge(
        classical_dates.assign(_key=1),
        on='_key'
    )
    .drop(columns='_key')
)

# Добавляем реальные продажи
classical_sample_full = classical_grid.merge(
    classical_sample_history[
        [
            'date',
            'store_nbr',
            'item_nbr',
            'unit_sales_clean'
        ]
    ],
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# Отсутствующие записи -> 0
classical_sample_full['unit_sales_clean'] = (
    classical_sample_full['unit_sales_clean']
    .fillna(0)
    .astype('float32')
)

# Создаём идентификатор временного ряда
classical_sample_full['unique_id'] = (
    classical_sample_full['store_nbr'].astype(str)
    + '_'
    + classical_sample_full['item_nbr'].astype(str)
)

# Формат StatsForecast
classical_sf = (
    classical_sample_full[
        [
            'unique_id',
            'date',
            'unit_sales_clean'
        ]
    ]
    .rename(
        columns={
            'date': 'ds',
            'unit_sales_clean': 'y'
        }
    )
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

print('Размер classical_sf:', classical_sf.shape)

print(
    'Количество рядов:',
    classical_sf['unique_id'].nunique()
)

print(
    'Количество дат:',
    classical_sf['ds'].nunique()
)

print(
    'NaN:',
    classical_sf['y'].isna().sum()
)

print(
    'Память:',
    f"{classical_sf.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

classical_sf.head()


Размер classical_sf: (450000, 3)
Количество рядов: 5000
Количество дат: 90
NaN: 0
Память: 30.23 MB


,unique_id,ds,y
0,10_1084437,2017-05-02,0.0
1,10_1084437,2017-05-03,1.0
2,10_1084437,2017-05-04,1.0
3,10_1084437,2017-05-05,0.0
4,10_1084437,2017-05-06,2.0


### 5.3 AutoETS

Запускаем AutoETS на этих 5000 рядах:

In [188]:
import time

ets_model_5000 = AutoETS(
    season_length=7
)

sf_ets_5000 = StatsForecast(
    models=[ets_model_5000],
    freq='D',
    n_jobs=-1
)

start_time = time.time()

ets_forecast_5000 = sf_ets_5000.forecast(
    df=classical_sf,
    h=FORECAST_HORIZON
)

ets_time_5000 = time.time() - start_time

print(
    f'Время AutoETS для 5000 рядов: '
    f'{ets_time_5000:.2f} секунд'
)

print(
    'Размер прогноза:',
    ets_forecast_5000.shape
)

print(
    'Количество рядов:',
    ets_forecast_5000['unique_id'].nunique()
)

print(
    'Количество прогнозных дат:',
    ets_forecast_5000['ds'].nunique()
)

print(
    'NaN в прогнозе:',
    ets_forecast_5000['AutoETS'].isna().sum()
)

ets_forecast_5000.head()

Время AutoETS для 5000 рядов: 100.21 секунд
Размер прогноза: (80000, 3)
Количество рядов: 5000
Количество прогнозных дат: 16
NaN в прогнозе: 0


,unique_id,ds,AutoETS
0,10_1084437,2017-07-31,0.722492
1,10_1084437,2017-08-01,0.722492
2,10_1084437,2017-08-02,0.722492
3,10_1084437,2017-08-03,0.722492
4,10_1084437,2017-08-04,0.722492


### 5.4 AutoTheta
Запускаем AutoTheta на этих 5000 рядах:

In [189]:
theta_model_5000 = AutoTheta(
    season_length=7
)

sf_theta_5000 = StatsForecast(
    models=[theta_model_5000],
    freq='D',
    n_jobs=-1
)

start_time = time.time()

theta_forecast_5000 = sf_theta_5000.forecast(
    df=classical_sf,
    h=FORECAST_HORIZON
)

theta_time_5000 = time.time() - start_time

print(
    f'Время AutoTheta для 5000 рядов: '
    f'{theta_time_5000:.2f} секунд'
)

print(
    'Размер прогноза:',
    theta_forecast_5000.shape
)

print(
    'Количество рядов:',
    theta_forecast_5000['unique_id'].nunique()
)

print(
    'Количество прогнозных дат:',
    theta_forecast_5000['ds'].nunique()
)

print(
    'NaN в прогнозе:',
    theta_forecast_5000['AutoTheta'].isna().sum()
)

theta_forecast_5000.head()

Время AutoTheta для 5000 рядов: 30.74 секунд
Размер прогноза: (80000, 3)
Количество рядов: 5000
Количество прогнозных дат: 16
NaN в прогнозе: 0


,unique_id,ds,AutoTheta
0,10_1084437,2017-07-31,0.894199
1,10_1084437,2017-08-01,0.896998
2,10_1084437,2017-08-02,0.899797
3,10_1084437,2017-08-03,0.902597
4,10_1084437,2017-08-04,0.905396


Сравним четыре baseline на одних и тех же 5000 рядах.

Собираем validation для 5000 рядов:

In [190]:
classical_validation = validation_full.merge(
    classical_sample_pairs,
    on=['store_nbr', 'item_nbr'],
    how='inner'
)

classical_validation['unique_id'] = (
    classical_validation['store_nbr'].astype(str)
    + '_'
    + classical_validation['item_nbr'].astype(str)
)

print(
    'Размер validation:',
    classical_validation.shape
)

print(
    'Количество рядов:',
    classical_validation['unique_id'].nunique()
)

print(
    'Количество дат:',
    classical_validation['date'].nunique()
)

print(
    'Ожидаемое количество наблюдений:',
    CLASSICAL_SAMPLE_SIZE * FORECAST_HORIZON
)

print(
    'NaN в target:',
    classical_validation['unit_sales_clean'].isna().sum()
)

classical_validation.head()

Размер validation: (80000, 5)
Количество рядов: 5000
Количество дат: 16
Ожидаемое количество наблюдений: 80000
NaN в target: 0


,store_nbr,item_nbr,date,unit_sales_clean,unique_id
0,1,123347,2017-07-31,0.0,1_123347
1,1,123347,2017-08-01,0.0,1_123347
2,1,123347,2017-08-02,3.0,1_123347
3,1,123347,2017-08-03,1.0,1_123347
4,1,123347,2017-08-04,0.0,1_123347


Объединяем прогнозы и считаем метрики:

In [191]:
# Начинаем с фактических значений
classical_compare = classical_validation.copy()

# Naive
classical_compare = classical_compare.merge(
    validation_naive[
        [
            'date',
            'store_nbr',
            'item_nbr',
            'naive_pred'
        ]
    ],
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# SeasonalNaive
classical_compare = classical_compare.merge(
    validation_seasonal[
        [
            'date',
            'store_nbr',
            'item_nbr',
            'seasonal_naive_pred'
        ]
    ],
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# AutoETS
ets_5000_for_merge = (
    ets_forecast_5000
    .rename(
        columns={
            'ds': 'date',
            'AutoETS': 'autoets_pred'
        }
    )
)

classical_compare = classical_compare.merge(
    ets_5000_for_merge[
        ['unique_id', 'date', 'autoets_pred']
    ],
    on=['unique_id', 'date'],
    how='left'
)

# AutoTheta
theta_5000_for_merge = (
    theta_forecast_5000
    .rename(
        columns={
            'ds': 'date',
            'AutoTheta': 'autotheta_pred'
        }
    )
)

classical_compare = classical_compare.merge(
    theta_5000_for_merge[
        ['unique_id', 'date', 'autotheta_pred']
    ],
    on=['unique_id', 'date'],
    how='left'
)

# Добавляем perishable
classical_compare = classical_compare.merge(
    items[['item_nbr', 'perishable']],
    on='item_nbr',
    how='left'
)

classical_compare['weight'] = np.where(
    classical_compare['perishable'] == 1,
    1.25,
    1.0
).astype('float32')


# Проверяем пропуски
prediction_columns = [
    'naive_pred',
    'seasonal_naive_pred',
    'autoets_pred',
    'autotheta_pred'
]

print('Количество NaN:')
print(
    classical_compare[
        ['unit_sales_clean'] + prediction_columns
    ].isna().sum()
)


# Функция для удобного сравнения
def evaluate_model(df, pred_col, model_name):
    return {
        'Model': model_name,

        'MAE': mae(
            df['unit_sales_clean'],
            df[pred_col]
        ),

        'NWRMSLE': nwrmsle(
            df['unit_sales_clean'],
            df[pred_col],
            weights=df['weight']
        )
    }

# Считаем метрики
classical_results = pd.DataFrame([
    evaluate_model(
        classical_compare,
        'naive_pred',
        'Naive'
    ),
    evaluate_model(
        classical_compare,
        'seasonal_naive_pred',
        'SeasonalNaive'
    ),
    evaluate_model(
        classical_compare,
        'autoets_pred',
        'AutoETS'
    ),
    evaluate_model(
        classical_compare,
        'autotheta_pred',
        'AutoTheta'
    )
])

classical_results = (
    classical_results
    .sort_values('NWRMSLE')
    .reset_index(drop=True)
)

print('\nРезультаты на 5000 рядах:')
display(classical_results)


Количество NaN:
unit_sales_clean       0
naive_pred             0
seasonal_naive_pred    0
autoets_pred           0
autotheta_pred         0
dtype: int64

Результаты на 5000 рядах:


,Model,MAE,NWRMSLE
0,AutoETS,3.576455,0.749299
1,AutoTheta,3.622445,0.751534
2,SeasonalNaive,4.071433,0.888736
3,Naive,5.153103,0.945362


На фиксированной выборке из 5000 рядов AutoETS и AutoTheta сравниваются с Naive и SeasonalNaive на одинаковых объектах и датах.

Naive и SeasonalNaive дополнительно оцениваются на полной validation-панели:

In [192]:
baseline_full_results = pd.DataFrame({
    'Model': [
        'Naive',
        'SeasonalNaive'
    ],
    'MAE': [
        naive_mae,
        seasonal_mae
    ],
    'NWRMSLE': [
        naive_nwrmsle,
        seasonal_nwrmsle
    ]
})

display(baseline_full_results)

,Model,MAE,NWRMSLE
0,Naive,5.252054,0.941351
1,SeasonalNaive,3.967178,0.879374


На полной validation SeasonalNaive заметно улучшает Naive, что подтверждает наличие недельной сезонности. На фиксированной выборке из 5000 рядов AutoETS и AutoTheta показывают более низкие ошибки, чем оба простых baseline. Однако результаты классических моделей относятся только к этой подвыборке, поэтому далее они используются как дополнительный ориентир, а не как оценка на всей validation-панели.

## 6. CatBoost

В  качестве основной ML-модели используем CatBoost. Для панельного временного ряда задача формулируется как direct multi-horizon forecasting: одна строка обучающей выборки соответствует конкретной паре магазин–товар и одному из 16 будущих горизонтов.

Исторические признаки для каждого forecast origin рассчитываются только по данным, доступным до даты train_end, что предотвращает утечку информации из будущего.

Сначала на фиксированном training origin проводится последовательный ablation-анализ нескольких конфигураций CatBoost. После выбора конфигурации финальная модель переобучается на нескольких historical origins.

In [193]:
!pip install catboost -q


In [194]:
from catboost import CatBoostRegressor

### 6.1 Формирование ML-выборки

Для первоначального выбора конфигурации CatBoost используется один training origin с train_end = 2017-07-14 и отдельный validation origin с train_end = 2017-07-30.

Горизонт прогнозирования составляет 16 дней: training target соответствует периоду 15.07.2017–30.07.2017, validation — 31.07.2017–15.08.2017.

Для каждой пары магазин–товар рассчитываются лаги 1, 7, 14 и 28 дней, а также средние продажи за последние 7, 14 и 28 календарных дней. Все исторические признаки строятся только по информации, доступной до соответствующего forecast origin.

Для CatBoost используется direct multi-horizon постановка: одна строка соответствует конкретной паре (store, item) и одному шагу горизонта от 1 до 16.

Исторические признаки рассчитываются только по данным, доступным на момент forecast origin. lag_1 соответствует последнему известному дню перед началом прогноза, lag_7 — значению за 7 дней до первого прогнозируемого дня и т.д. Rolling-признаки также рассчитываются только по истории до forecast origin, поэтому данные из target-периода в признаки не попадают.

In [8]:
import pandas as pd
import numpy as np
import gc
import time

from pathlib import Path

DATA_PATH = Path(
    '/kaggle/input/datasets/m2101119/'
    'favorita-grocery-sales-forecasting'
)

train_path = DATA_PATH / 'train.csv'

# Небольшие вспомогательные таблицы
items = pd.read_csv(
    DATA_PATH / 'items.csv',
    dtype={
        'item_nbr': 'int32',
        'class': 'int16',
        'perishable': 'int8'
    }
)

stores = pd.read_csv(
    DATA_PATH / 'stores.csv',
    dtype={
        'store_nbr': 'int16',
        'cluster': 'int8'
    }
)

test = pd.read_csv(
    DATA_PATH / 'test.csv',
    parse_dates=['date'],
    dtype={
        'id': 'int32',
        'store_nbr': 'int16',
        'item_nbr': 'int32'
    },
    low_memory=False
)

# Основные даты задачи
FORECAST_HORIZON = 16

VAL_START = pd.Timestamp('2017-07-31')
VAL_END = pd.Timestamp('2017-08-15')

ML_DATA_START = pd.Timestamp('2017-05-01')
ML_DATA_END = VAL_END

print('items:', items.shape)
print('stores:', stores.shape)
print('test:', test.shape)

print(
    '\nTest period:',
    test['date'].min().date(),
    '—',
    test['date'].max().date()
)

print(
    'Validation:',
    VAL_START.date(),
    '—',
    VAL_END.date()
)

print(
    'Train.csv существует:',
    train_path.exists()
)

items: (4100, 4)
stores: (54, 5)
test: (3370464, 5)

Test period: 2017-08-16 — 2017-08-31
Validation: 2017-07-31 — 2017-08-15
Train.csv существует: True


In [9]:
# Загружаем только необходимый для ML период train.csv

ml_parts = []

for chunk in pd.read_csv(
    train_path,
    usecols=[
        'date',
        'store_nbr',
        'item_nbr',
        'unit_sales',
        'onpromotion'
    ],
    chunksize=1_000_000,
    dtype={
        'store_nbr': 'int16',
        'item_nbr': 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date'],
    low_memory=False
):
    part = chunk[
        (chunk['date'] >= ML_DATA_START) &
        (chunk['date'] <= ML_DATA_END)
    ].copy()

    if len(part) > 0:
        ml_parts.append(part)

ml_data = pd.concat(
    ml_parts,
    ignore_index=True
)

del ml_parts
gc.collect()

# Отрицательные продажи соответствуют возвратам.
# Для прогнозирования спроса приводим их к нулю.
ml_data['unit_sales_clean'] = (
    ml_data['unit_sales']
    .clip(lower=0)
    .astype('float32')
)

ml_data['onpromotion'] = (
    ml_data['onpromotion']
    .fillna(False)
    .astype(bool)
)

print('ml_data:', ml_data.shape)
print(
    'Период:',
    ml_data['date'].min().date(),
    '—',
    ml_data['date'].max().date()
)
print(
    'Память:',
    f'{ml_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

ml_data: (11320790, 6)
Период: 2017-05-01 — 2017-08-15
Память: 248.32 MB


In [12]:
def build_ml_origin(
    ml_data,
    train_end,
    forecast_horizon=16,
    active_days=16
):
    """
    Создаёт supervised dataset для одного forecast origin.

    Все исторические признаки строятся только
    по данным <= train_end.
    """

    train_end = pd.Timestamp(train_end)

    forecast_start = (
        train_end + pd.Timedelta(days=1)
    )

    forecast_end = (
        forecast_start
        + pd.Timedelta(days=forecast_horizon - 1)
    )


    # Определяем universe store-item только по доступной истории
    active_start = (
        train_end
        - pd.Timedelta(days=active_days - 1)
    )

    pairs = (
        ml_data[
            (ml_data['date'] >= active_start) &
            (ml_data['date'] <= train_end)
        ][['store_nbr', 'item_nbr']]
        .drop_duplicates()
        .reset_index(drop=True)
    )


    # Исторические признаки
    features = pairs.copy()

    # Наш lag_1 = последний известный день
    for lag in [1, 7, 14, 28]:

        lag_date = (
            train_end
            - pd.Timedelta(days=lag - 1)
        )

        lag_values = (
            ml_data[
                ml_data['date'] == lag_date
            ][
                [
                    'store_nbr',
                    'item_nbr',
                    'unit_sales_clean'
                ]
            ]
            .rename(
                columns={
                    'unit_sales_clean': f'lag_{lag}'
                }
            )
        )

        features = features.merge(
            lag_values,
            on=['store_nbr', 'item_nbr'],
            how='left'
        )

        features[f'lag_{lag}'] = (
            features[f'lag_{lag}']
            .fillna(0)
            .astype('float32')
        )

    # Rolling means
    for window in [7, 14, 28]:

        window_start = (
            train_end
            - pd.Timedelta(days=window - 1)
        )

        window_data = ml_data[
            (ml_data['date'] >= window_start) &
            (ml_data['date'] <= train_end)
        ][
            [
                'store_nbr',
                'item_nbr',
                'unit_sales_clean'
            ]
        ]

        rolling_sum = (
            window_data
            .groupby(
                ['store_nbr', 'item_nbr'],
                as_index=False
            )['unit_sales_clean']
            .sum()
            .rename(
                columns={
                    'unit_sales_clean':
                    f'rolling_sum_{window}'
                }
            )
        )

        features = features.merge(
            rolling_sum,
            on=['store_nbr', 'item_nbr'],
            how='left'
        )

        features[f'rolling_sum_{window}'] = (
            features[f'rolling_sum_{window}']
            .fillna(0)
            .astype('float32')
        )

        features[f'rolling_mean_{window}'] = (
            features[f'rolling_sum_{window}']
            / window
        ).astype('float32')


    # Создаём 16 будущих дат
    forecast_dates = pd.DataFrame({
        'date': pd.date_range(
            forecast_start,
            forecast_end,
            freq='D'
        )
    })

    forecast_dates['horizon'] = np.arange(
        1,
        forecast_horizon + 1,
        dtype=np.int8
    )

    pairs_grid = (
        pairs.assign(_key=1)
        .merge(
            forecast_dates.assign(_key=1),
            on='_key'
        )
        .drop(columns='_key')
    )


    # Target
    actual = ml_data[
        (ml_data['date'] >= forecast_start) &
        (ml_data['date'] <= forecast_end)
    ][
        [
            'date',
            'store_nbr',
            'item_nbr',
            'unit_sales_clean'
        ]
    ]

    result = pairs_grid.merge(
        actual,
        on=['date', 'store_nbr', 'item_nbr'],
        how='left'
    )

    result['target'] = (
        result['unit_sales_clean']
        .fillna(0)
        .astype('float32')
    )

    result = result.drop(
        columns='unit_sales_clean'
    )


    # Добавляем исторические признаки
    feature_cols = [
        'lag_1',
        'lag_7',
        'lag_14',
        'lag_28',
        'rolling_mean_7',
        'rolling_mean_14',
        'rolling_mean_28'
    ]

    result = result.merge(
        features[
            ['store_nbr', 'item_nbr'] + feature_cols
        ],
        on=['store_nbr', 'item_nbr'],
        how='left'
    )


    # Календарные признаки
    result['day_of_week'] = (
        result['date'].dt.dayofweek.astype('int8')
    )

    result['day_of_month'] = (
        result['date'].dt.day.astype('int8')
    )

    result['month'] = (
        result['date'].dt.month.astype('int8')
    )

    result['is_weekend'] = (
        result['day_of_week']
        .isin([5, 6])
        .astype('int8')
    )

    # Сохраняем cutoff для контроля
    result['train_end'] = train_end

    return result

In [13]:
ML_TRAIN_END = pd.Timestamp('2017-07-14')
ML_VAL_TRAIN_END = pd.Timestamp('2017-07-30')

# Training origin:
# пары определяются только по истории до forecast origin
ml_train = build_ml_origin(
    ml_data=ml_data,
    train_end=ML_TRAIN_END,
    forecast_horizon=FORECAST_HORIZON,
    active_days=16
)

# Validation:
# пока строим по той же временно корректной схеме
ml_val = build_ml_origin(
    ml_data=ml_data,
    train_end=ML_VAL_TRAIN_END,
    forecast_horizon=FORECAST_HORIZON,
    active_days=16
)

gc.collect()

print('TRAIN')
print('Размер:', ml_train.shape)
print(
    'Период target:',
    ml_train['date'].min().date(),
    '—',
    ml_train['date'].max().date()
)
print(
    'Пар:',
    ml_train[['store_nbr', 'item_nbr']]
    .drop_duplicates()
    .shape[0]
)
print(
    'Память:',
    f'{ml_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print('\nVALIDATION')
print('Размер:', ml_val.shape)
print(
    'Период target:',
    ml_val['date'].min().date(),
    '—',
    ml_val['date'].max().date()
)
print(
    'Пар:',
    ml_val[['store_nbr', 'item_nbr']]
    .drop_duplicates()
    .shape[0]
)
print(
    'Память:',
    f'{ml_val.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

# Дополнительная проверка
print('\nCHECK')
print(
    'Train target начинается после cutoff:',
    ml_train['date'].min() > ML_TRAIN_END
)
print(
    'Validation target начинается после cutoff:',
    ml_val['date'].min() > ML_VAL_TRAIN_END
)
print(
    'Train заканчивается до начала validation:',
    ml_train['date'].max() < ml_val['date'].min()
)

TRAIN
Размер: (2366720, 17)
Период target: 2017-07-15 — 2017-07-30
Пар: 147920
Память: 133.17 MB

VALIDATION
Размер: (2358096, 17)
Период target: 2017-07-31 — 2017-08-15
Пар: 147381
Память: 132.68 MB

CHECK
Train target начинается после cutoff: True
Validation target начинается после cutoff: True
Train заканчивается до начала validation: True


### 6.2 CatBoost v1: базовая модель

В первой версии используем только календарные и исторические признаки. Модель обучается непосредственно на исходном target.

Эта версия служит отправной точкой для последующих экспериментов.

In [199]:
CATBOOST_V1_FEATURES = [
    'horizon',
    'day_of_week',
    'day_of_month',
    'month',
    'is_weekend',

    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',

    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_28'
]

X_train = ml_train[CATBOOST_V1_FEATURES]
y_train = ml_train['target']

X_val = ml_val[CATBOOST_V1_FEATURES]
y_val = ml_val['target']

print('X_train:', X_train.shape)
print('X_val:', X_val.shape)

print('\nТипы:')
print(X_train.dtypes)

print(
    '\nПамять X_train:',
    f'{X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    'Память X_val:',
    f'{X_val.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    '\nNaN train:',
    X_train.isna().sum().sum()
)

print(
    'NaN validation:',
    X_val.isna().sum().sum()
)

X_train: (2366720, 12)
X_val: (2358096, 12)

Типы:
horizon               int8
day_of_week           int8
day_of_month          int8
month                 int8
is_weekend            int8
lag_1              float32
lag_7              float32
lag_14             float32
lag_28             float32
rolling_mean_7     float32
rolling_mean_14    float32
rolling_mean_28    float32
dtype: object

Память X_train: 74.48 MB
Память X_val: 74.21 MB

NaN train: 0
NaN validation: 0


In [51]:
from catboost import CatBoostRegressor
import time

catboost_v1 = CatBoostRegressor(
    iterations=300,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=25,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v1.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=30,
    use_best_model=True
)

catboost_v1_time = time.time() - start_time

print(
    f'\nВремя обучения: '
    f'{catboost_v1_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v1.get_best_iteration()
)

print(
    'Лучший validation RMSE:',
    catboost_v1.get_best_score()['validation']['RMSE']
)


NameError: name 'X_train' is not defined

Считаем MAE и NWRMSLE CatBoost v1:

In [201]:
# Прогноз
catboost_v1_pred = catboost_v1.predict(X_val)

print(
    'Минимальный прогноз ДО clipping:',
    catboost_v1_pred.min()
)

print(
    'Максимальный прогноз ДО clipping:',
    catboost_v1_pred.max()
)

# Продажи не могут быть отрицательными
catboost_v1_pred = np.clip(
    catboost_v1_pred,
    0,
    None
)

# Добавляем прогноз в validation
ml_val['catboost_v1_pred'] = (
    catboost_v1_pred.astype('float32')
)

# Добавляем perishable для весов NWRMSLE
ml_val_eval = ml_val.merge(
    items[['item_nbr', 'perishable']],
    on='item_nbr',
    how='left'
)

ml_val_eval['weight'] = np.where(
    ml_val_eval['perishable'] == 1,
    1.25,
    1.0
).astype('float32')

print(
    '\nПропусков в perishable:',
    ml_val_eval['perishable'].isna().sum()
)

# Метрики
catboost_v1_mae = np.mean(
    np.abs(
        ml_val_eval['target'].values
        - ml_val_eval['catboost_v1_pred'].values
    )
)

catboost_v1_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(
                ml_val_eval['catboost_v1_pred'].values
            )
            -
            np.log1p(
                ml_val_eval['target'].values
            )
        ) ** 2,
        weights=ml_val_eval['weight'].values
    )
)

print(
    f'\nCatBoost v1 MAE:     '
    f'{catboost_v1_mae:.4f}'
)

print(
    f'CatBoost v1 NWRMSLE: '
    f'{catboost_v1_nwrmsle:.4f}'
)

Минимальный прогноз ДО clipping: -0.8363636271190842
Максимальный прогноз ДО clipping: 897.3748301007795

Пропусков в perishable: 0

CatBoost v1 MAE:     3.5793
CatBoost v1 NWRMSLE: 0.8209


### 6.3 CatBoost v2: логарифмирование target

В первой версии модель обучалась на исходных продажах. Распределение target сильно скошено, поэтому во второй версии обучаю CatBoost на log1p(target), а перед расчётом MAE возвращаю прогноз в исходную шкалу через expm1:

In [202]:
# Логарифмируем target
y_train_log = np.log1p(
    ml_train['target'].values
).astype('float32')

y_val_log = np.log1p(
    ml_val['target'].values
).astype('float32')

catboost_v2 = CatBoostRegressor(
    iterations=300,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=25,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v2.fit(
    X_train,
    y_train_log,
    eval_set=(X_val, y_val_log),
    early_stopping_rounds=30,
    use_best_model=True
)

catboost_v2_time = time.time() - start_time

print(
    f'\nВремя обучения CatBoost v2: '
    f'{catboost_v2_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v2.get_best_iteration()
)

print(
    'Лучший validation RMSE в log-шкале:',
    catboost_v2.get_best_score()['validation']['RMSE']
)

0:	learn: 1.0173368	test: 1.0106358	best: 1.0106358 (0)	total: 235ms	remaining: 1m 10s
25:	learn: 0.6830775	test: 0.6878414	best: 0.6878414 (25)	total: 5.33s	remaining: 56.2s
50:	learn: 0.6703596	test: 0.6798828	best: 0.6798828 (50)	total: 10.4s	remaining: 50.7s
75:	learn: 0.6682739	test: 0.6792839	best: 0.6791516 (69)	total: 15.3s	remaining: 45.1s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.6791515732
bestIteration = 69

Shrink model to first 70 iterations.

Время обучения CatBoost v2: 0.35 минут
Лучшая итерация: 69
Лучший validation RMSE в log-шкале: 0.6791515731995688


Считаем метрики CatBoost v2

In [203]:
# Прогноз в log-шкале
catboost_v2_pred_log = catboost_v2.predict(X_val)

# Возвращаемся в исходную шкалу продаж
catboost_v2_pred = np.expm1(
    catboost_v2_pred_log
)

print(
    'Минимальный прогноз ДО clipping:',
    catboost_v2_pred.min()
)

print(
    'Максимальный прогноз ДО clipping:',
    catboost_v2_pred.max()
)

# Продажи не могут быть отрицательными
catboost_v2_pred = np.clip(
    catboost_v2_pred,
    0,
    None
)

# Добавляем прогноз
ml_val_eval['catboost_v2_pred'] = (
    catboost_v2_pred.astype('float32')
)


# MAE
catboost_v2_mae = np.mean(
    np.abs(
        ml_val_eval['target'].values
        - ml_val_eval['catboost_v2_pred'].values
    )
)


# NWRMSLE
catboost_v2_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(
                ml_val_eval['catboost_v2_pred'].values
            )
            -
            np.log1p(
                ml_val_eval['target'].values
            )
        ) ** 2,
        weights=ml_val_eval['weight'].values
    )
)

print(
    f'\nCatBoost v2 MAE:     '
    f'{catboost_v2_mae:.4f}'
)

print(
    f'CatBoost v2 NWRMSLE: '
    f'{catboost_v2_nwrmsle:.4f}'
)


# Сравнение v1 и v2
comparison_v1_v2 = pd.DataFrame({
    'Model': [
        'CatBoost v1 — raw target',
        'CatBoost v2 — log1p target'
    ],
    'MAE': [
        catboost_v1_mae,
        catboost_v2_mae
    ],
    'NWRMSLE': [
        catboost_v1_nwrmsle,
        catboost_v2_nwrmsle
    ]
})

display(comparison_v1_v2)


Минимальный прогноз ДО clipping: -0.13604984793395067
Максимальный прогноз ДО clipping: 291.08138083327367

CatBoost v2 MAE:     3.2099
CatBoost v2 NWRMSLE: 0.6797


,Model,MAE,NWRMSLE
0,CatBoost v1 — raw target,3.579256,0.820891
1,CatBoost v2 — log1p target,3.209863,0.679722


Улучшились обе метрики. Относительно CatBoost v1 NWRMSLE снизился примерно на 16.6%, а MAE - примерно на 10.9%. Относительно SeasonalNaive снижение NWRMSLE уже около 23%.

Функция обучения стала гораздо лучше согласована с основной метрикой, а логарифмирование одновременно уменьшило влияние экстремальных продаж.

Сохраняем CatBoost v2 и текущие результаты:

### 6.4 CatBoost v3: акции и метаданные

В следующей версии добавляем известный на дату прогноза признак onpromotion, а также статические характеристики товара и магазина. Это позволяет модели учитывать различия между категориями товаров и типами магазинов, не опираясь только на недавнюю историю продаж.

In [204]:
promo_data = ml_data[
    [
        'date',
        'store_nbr',
        'item_nbr',
        'onpromotion'
    ]
].copy()

# Добавляем promotion в train
ml_train_v3 = ml_train.merge(
    promo_data,
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# Добавляем promotion в validation
ml_val_v3 = ml_val.merge(
    promo_data,
    on=['date', 'store_nbr', 'item_nbr'],
    how='left'
)

# Если строки продажи в исходном train не было, onpromotion после merge будет NaN.
# Для текущей постановки заполняем False.
ml_train_v3['onpromotion'] = (
    ml_train_v3['onpromotion']
    .fillna(False)
    .astype('int8')
)

ml_val_v3['onpromotion'] = (
    ml_val_v3['onpromotion']
    .fillna(False)
    .astype('int8')
)

print('TRAIN')
print('Размер:', ml_train_v3.shape)
print(
    'Promotion = 1:',
    f"{ml_train_v3['onpromotion'].sum():,}"
)
print(
    'Доля promotion:',
    f"{ml_train_v3['onpromotion'].mean() * 100:.2f}%"
)

print('\nVALIDATION')
print('Размер:', ml_val_v3.shape)
print(
    'Promotion = 1:',
    f"{ml_val_v3['onpromotion'].sum():,}"
)
print(
    'Доля promotion:',
    f"{ml_val_v3['onpromotion'].mean() * 100:.2f}%"
)

print(
    '\nNaN train:',
    ml_train_v3['onpromotion'].isna().sum()
)
print(
    'NaN validation:',
    ml_val_v3['onpromotion'].isna().sum()
)

/tmp/ipykernel_58/4269462928.py:28: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



TRAIN
Размер: (2366720, 18)
Promotion = 1: 188,612
Доля promotion: 7.97%

VALIDATION
Размер: (2358096, 19)
Promotion = 1: 167,007
Доля promotion: 7.08%

NaN train: 0
NaN validation: 0


/tmp/ipykernel_58/4269462928.py:34: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Добавляем характеристики товаров и магазинов.

Для товара у нас есть: family - категория товара; class - класс товара; perishable - скоропортящийся ли товар.

Для магазина: city, state, type, cluster.

Это уже делает модель намного содержательнее. Например, два товара с одинаковыми последними продажами могут принадлежать совершенно разным категориям, а одинаковый товар может вести себя по-разному в магазинах разных типов.

In [205]:
# Добавляем item metadata
item_features = items[
    [
        'item_nbr',
        'family',
        'class',
        'perishable'
    ]
].copy()

ml_train_v3 = ml_train_v3.merge(
    item_features,
    on='item_nbr',
    how='left'
)

ml_val_v3 = ml_val_v3.merge(
    item_features,
    on='item_nbr',
    how='left'
)



# Добавляем store metadata
store_features = stores[
    [
        'store_nbr',
        'city',
        'state',
        'type',
        'cluster'
    ]
].copy()

ml_train_v3 = ml_train_v3.merge(
    store_features,
    on='store_nbr',
    how='left'
)

ml_val_v3 = ml_val_v3.merge(
    store_features,
    on='store_nbr',
    how='left'
)


# Проверяем результат
NEW_FEATURES = [
    'onpromotion',
    'family',
    'class',
    'perishable',
    'city',
    'state',
    'type',
    'cluster'
]

print('TRAIN:', ml_train_v3.shape)
print('VALIDATION:', ml_val_v3.shape)

print('\nNaN в новых признаках TRAIN:')
print(
    ml_train_v3[NEW_FEATURES]
    .isna()
    .sum()
)

print('\nNaN в новых признаках VALIDATION:')
print(
    ml_val_v3[NEW_FEATURES]
    .isna()
    .sum()
)

print('\nПример:')
display(
    ml_train_v3[
        [
            'store_nbr',
            'item_nbr',
            'date',
            'onpromotion',
            'family',
            'class',
            'perishable',
            'city',
            'state',
            'type',
            'cluster',
            'target'
        ]
    ].head(10)
)


TRAIN: (2366720, 25)
VALIDATION: (2358096, 26)

NaN в новых признаках TRAIN:
onpromotion    0
family         0
class          0
perishable     0
city           0
state          0
type           0
cluster        0
dtype: int64

NaN в новых признаках VALIDATION:
onpromotion    0
family         0
class          0
perishable     0
city           0
state          0
type           0
cluster        0
dtype: int64

Пример:


,store_nbr,item_nbr,date,onpromotion,family,class,perishable,city,state,type,cluster,target
0,1,96995,2017-07-15,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
1,1,96995,2017-07-16,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
2,1,96995,2017-07-17,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
3,1,96995,2017-07-18,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
4,1,96995,2017-07-19,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
5,1,96995,2017-07-20,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
6,1,96995,2017-07-21,0,GROCERY I,1093,0,Quito,Pichincha,D,13,3.0
7,1,96995,2017-07-22,0,GROCERY I,1093,0,Quito,Pichincha,D,13,2.0
8,1,96995,2017-07-23,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0
9,1,96995,2017-07-24,0,GROCERY I,1093,0,Quito,Pichincha,D,13,0.0


Готовим признаки CatBoost v3.

Добавляем к нашим 12 признакам: onpromotion, family, class, perishable, city, state, type, cluster

Для строковых категорий создадим одинаковые коды для train и validation

In [206]:
# Копируем только нужные новые признаки
# Кодируем строковые категории одинаково для train и validation

categorical_text_cols = [
    'family',
    'city',
    'state',
    'type'
]

for col in categorical_text_cols:

    # Все возможные категории берем из metadata, поэтому train/validation получают одну систему кодов
    if col == 'family':
        categories = sorted(items[col].dropna().unique())
    else:
        categories = sorted(stores[col].dropna().unique())

    mapping = {
        value: idx
        for idx, value in enumerate(categories)
    }

    ml_train_v3[col + '_code'] = (
        ml_train_v3[col]
        .map(mapping)
        .astype('int16')
    )

    ml_val_v3[col + '_code'] = (
        ml_val_v3[col]
        .map(mapping)
        .astype('int16')
    )


# Полный набор признаков v3
CATBOOST_V3_FEATURES = [
    # признаки v2
    'horizon',
    'day_of_week',
    'day_of_month',
    'month',
    'is_weekend',

    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',

    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_28',

    # новые признаки
    'onpromotion',

    'family_code',
    'class',
    'perishable',

    'city_code',
    'state_code',
    'type_code',
    'cluster'
]


X_train_v3 = ml_train_v3[CATBOOST_V3_FEATURES]
X_val_v3 = ml_val_v3[CATBOOST_V3_FEATURES]


print('X_train_v3:', X_train_v3.shape)
print('X_val_v3:', X_val_v3.shape)

print('\nКоличество признаков:', len(CATBOOST_V3_FEATURES))

print('\nТипы:')
print(X_train_v3.dtypes)

print(
    '\nПамять X_train_v3:',
    f'{X_train_v3.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    'Память X_val_v3:',
    f'{X_val_v3.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    '\nNaN train:',
    X_train_v3.isna().sum().sum()
)

print(
    'NaN validation:',
    X_val_v3.isna().sum().sum()
)


X_train_v3: (2366720, 20)
X_val_v3: (2358096, 20)

Количество признаков: 20

Типы:
horizon               int8
day_of_week           int8
day_of_month          int8
month                 int8
is_weekend            int8
lag_1              float32
lag_7              float32
lag_14             float32
lag_28             float32
rolling_mean_7     float32
rolling_mean_14    float32
rolling_mean_28    float32
onpromotion           int8
family_code          int16
class                int16
perishable            int8
city_code            int16
state_code           int16
type_code            int16
cluster               int8
dtype: object

Память X_train_v3: 103.83 MB
Память X_val_v3: 103.45 MB

NaN train: 0
NaN validation: 0


Обучаем CatBoost v3:

In [207]:
catboost_v3 = CatBoostRegressor(
    iterations=400,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=25,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v3.fit(
    X_train_v3,
    y_train_log,
    eval_set=(X_val_v3, y_val_log),
    early_stopping_rounds=30,
    use_best_model=True
)

catboost_v3_time = time.time() - start_time

print(
    f'\nВремя обучения CatBoost v3: '
    f'{catboost_v3_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v3.get_best_iteration()
)

print(
    'Лучший validation RMSE в log-шкале:',
    catboost_v3.get_best_score()['validation']['RMSE']
)

0:	learn: 1.0159804	test: 1.0103321	best: 1.0103321 (0)	total: 254ms	remaining: 1m 41s
25:	learn: 0.6654893	test: 0.6762868	best: 0.6762868 (25)	total: 6.02s	remaining: 1m 26s
50:	learn: 0.6495336	test: 0.6635892	best: 0.6635892 (50)	total: 11.7s	remaining: 1m 20s
75:	learn: 0.6461090	test: 0.6621122	best: 0.6621122 (75)	total: 17.1s	remaining: 1m 13s
100:	learn: 0.6440040	test: 0.6611549	best: 0.6611549 (100)	total: 22.7s	remaining: 1m 7s
125:	learn: 0.6422462	test: 0.6602299	best: 0.6602299 (125)	total: 28.2s	remaining: 1m 1s
150:	learn: 0.6407206	test: 0.6594230	best: 0.6593695 (149)	total: 33.7s	remaining: 55.6s
175:	learn: 0.6393882	test: 0.6587014	best: 0.6587014 (175)	total: 39.4s	remaining: 50.2s
200:	learn: 0.6380824	test: 0.6581833	best: 0.6581833 (200)	total: 45.1s	remaining: 44.6s
225:	learn: 0.6369851	test: 0.6577990	best: 0.6577893 (222)	total: 50.8s	remaining: 39.1s
250:	learn: 0.6359648	test: 0.6572769	best: 0.6572769 (250)	total: 56.7s	remaining: 33.7s
275:	learn: 0.63

Считаем MAE и NWRMSLE CatBoost v3:

In [208]:
# Прогноз CatBoost v3
catboost_v3_pred_log = catboost_v3.predict(
    X_val_v3
)

# Обратное преобразование log1p
catboost_v3_pred = np.expm1(
    catboost_v3_pred_log
)

print(
    'Минимальный прогноз ДО clipping:',
    catboost_v3_pred.min()
)

print(
    'Максимальный прогноз ДО clipping:',
    catboost_v3_pred.max()
)

# Продажи не могут быть отрицательными
catboost_v3_pred = np.clip(
    catboost_v3_pred,
    0,
    None
)

# MAE
catboost_v3_mae = np.mean(
    np.abs(
        ml_val_eval['target'].values
        - catboost_v3_pred
    )
)


# NWRMSLE
catboost_v3_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(catboost_v3_pred)
            -
            np.log1p(
                ml_val_eval['target'].values
            )
        ) ** 2,
        weights=ml_val_eval['weight'].values
    )
)

print(
    f'\nCatBoost v3 MAE:     '
    f'{catboost_v3_mae:.4f}'
)

print(
    f'CatBoost v3 NWRMSLE: '
    f'{catboost_v3_nwrmsle:.4f}'
)


# Сравниваем v1, v2 и v3
catboost_comparison = pd.DataFrame({
    'Model': [
        'CatBoost v1 — raw target',
        'CatBoost v2 — log1p target',
        'CatBoost v3 — log1p + promo + metadata'
    ],
    'MAE': [
        catboost_v1_mae,
        catboost_v2_mae,
        catboost_v3_mae
    ],
    'NWRMSLE': [
        catboost_v1_nwrmsle,
        catboost_v2_nwrmsle,
        catboost_v3_nwrmsle
    ]
})

display(
    catboost_comparison.sort_values(
        'NWRMSLE'
    )
)


Минимальный прогноз ДО clipping: -0.42541330836340663
Максимальный прогноз ДО clipping: 543.4098136884754

CatBoost v3 MAE:     3.1121
CatBoost v3 NWRMSLE: 0.6563


,Model,MAE,NWRMSLE
2,CatBoost v3 — log1p + promo + metadata,3.112142,0.656276
1,CatBoost v2 — log1p target,3.209863,0.679722
0,CatBoost v1 — raw target,3.579256,0.820891


CatBoost v3 действительно улучшил обе метрики: NWRMSLE снизилась примерно на 3.3% относительно v2, МАЕ тоже немного снизилась.

Относительно SeasonalNaive по NWRMSLE улучшение уже примерно 25.5%. На выбранной validation добавление promotion и характеристик товаров и магазинов улучшило качество модели, а не просто увеличило число признаков.

Теперь следующий шаг - согласовать обучение с нашей weighted-метрикой. Сейчас NWRMSLE сильнее учитывает скоропортящиеся товары. Но CatBoost при обучении пока считает каждую строку одинаково.

CatBoost v4 - добавляем веса perishable:

### 6.5 CatBoost v4: веса скоропортящихся товаров

В метрике соревнования скоропортящиеся товары имеют вес 1.25, остальные - 1.0.

В v4 те же веса используем при обучении модели, чтобы objective лучше соответствовал основной метрике оценки:

In [209]:
# Веса для train и validation
train_weights_v4 = np.where(
    ml_train_v3['perishable'].values == 1,
    1.25,
    1.0
).astype('float32')

val_weights_v4 = np.where(
    ml_val_v3['perishable'].values == 1,
    1.25,
    1.0
).astype('float32')

print('TRAIN weights:')
print(
    pd.Series(train_weights_v4)
    .value_counts()
    .sort_index()
)

print('\nVALIDATION weights:')
print(
    pd.Series(val_weights_v4)
    .value_counts()
    .sort_index()
)

print(
    '\nСредний вес train:',
    train_weights_v4.mean()
)

print(
    'Средний вес validation:',
    val_weights_v4.mean()
)

TRAIN weights:
1.00    1839088
1.25     527632
Name: count, dtype: int64

VALIDATION weights:
1.00    1830848
1.25     527248
Name: count, dtype: int64

Средний вес train: 1.0557345
Средний вес validation: 1.0558976


In [210]:
catboost_v4 = CatBoostRegressor(
    iterations=500,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=25,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v4.fit(
    X_train_v3,
    y_train_log,

    sample_weight=train_weights_v4,

    eval_set=(X_val_v3, y_val_log),
    early_stopping_rounds=30,
    use_best_model=True
)

catboost_v4_time = time.time() - start_time

print(
    f'\nВремя обучения CatBoost v4: '
    f'{catboost_v4_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v4.get_best_iteration()
)

print(
    'Лучший validation RMSE:',
    catboost_v4.get_best_score()['validation']['RMSE']
)

0:	learn: 1.0210127	test: 1.0107193	best: 1.0107193 (0)	total: 261ms	remaining: 2m 10s
25:	learn: 0.6665631	test: 0.6763210	best: 0.6763210 (25)	total: 5.9s	remaining: 1m 47s
50:	learn: 0.6505637	test: 0.6639275	best: 0.6639275 (50)	total: 11.5s	remaining: 1m 41s
75:	learn: 0.6469819	test: 0.6621532	best: 0.6621532 (75)	total: 17s	remaining: 1m 34s
100:	learn: 0.6447543	test: 0.6609268	best: 0.6609268 (100)	total: 22.7s	remaining: 1m 29s
125:	learn: 0.6430139	test: 0.6600939	best: 0.6600848 (124)	total: 28.2s	remaining: 1m 23s
150:	learn: 0.6415286	test: 0.6596353	best: 0.6596207 (149)	total: 33.8s	remaining: 1m 18s
175:	learn: 0.6402243	test: 0.6591844	best: 0.6591824 (174)	total: 39.4s	remaining: 1m 12s
200:	learn: 0.6390767	test: 0.6587620	best: 0.6587620 (200)	total: 45.1s	remaining: 1m 7s
225:	learn: 0.6378820	test: 0.6582868	best: 0.6582868 (225)	total: 51s	remaining: 1m 1s
250:	learn: 0.6368582	test: 0.6578777	best: 0.6578777 (250)	total: 56.6s	remaining: 56.1s
275:	learn: 0.635

Считаем метрики CatBoost v4:

In [211]:
# Прогноз CatBoost v4
catboost_v4_pred_log = catboost_v4.predict(
    X_val_v3
)

# Возвращаем прогноз в исходную шкалу
catboost_v4_pred = np.expm1(
    catboost_v4_pred_log
)

print(
    'Минимальный прогноз ДО clipping:',
    catboost_v4_pred.min()
)

print(
    'Максимальный прогноз ДО clipping:',
    catboost_v4_pred.max()
)

# Отрицательные прогнозы заменяем на 0
catboost_v4_pred = np.clip(
    catboost_v4_pred,
    0,
    None
)


# MAE
catboost_v4_mae = np.mean(
    np.abs(
        ml_val_eval['target'].values
        - catboost_v4_pred
    )
)



# NWRMSLE
catboost_v4_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(catboost_v4_pred)
            -
            np.log1p(
                ml_val_eval['target'].values
            )
        ) ** 2,
        weights=ml_val_eval['weight'].values
    )
)


print(
    f'\nCatBoost v4 MAE:     '
    f'{catboost_v4_mae:.4f}'
)

print(
    f'CatBoost v4 NWRMSLE: '
    f'{catboost_v4_nwrmsle:.4f}'
)



# Сравнение всех CatBoost
catboost_comparison = pd.DataFrame({
    'Model': [
        'CatBoost v1 — raw target',
        'CatBoost v2 — log1p target',
        'CatBoost v3 — log1p + promo + metadata',
        'CatBoost v4 — v3 + perishable weights'
    ],

    'MAE': [
        catboost_v1_mae,
        catboost_v2_mae,
        catboost_v3_mae,
        catboost_v4_mae
    ],

    'NWRMSLE': [
        catboost_v1_nwrmsle,
        catboost_v2_nwrmsle,
        catboost_v3_nwrmsle,
        catboost_v4_nwrmsle
    ]
})

display(
    catboost_comparison.sort_values(
        'NWRMSLE'
    )
)

Минимальный прогноз ДО clipping: -0.4375709940735645
Максимальный прогноз ДО clipping: 533.5216395255333

CatBoost v4 MAE:     3.1118
CatBoost v4 NWRMSLE: 0.6562


,Model,MAE,NWRMSLE
3,CatBoost v4 — v3 + perishable weights,3.111792,0.656179
2,CatBoost v3 — log1p + promo + metadata,3.112142,0.656276
1,CatBoost v2 — log1p target,3.209863,0.679722
0,CatBoost v1 — raw target,3.579256,0.820891


v4 действительно улучшил v3, использование весов дало небольшое дополнительное улучшение NWRMSLE.Но основной скачок качества дали именно log1p и дополнительные признаки.

Теперь переходим к внешним признакам. Подготовим нефть и праздники и присоединим их:

### 6.6 CatBoost v5: внешние признаки

Последний эксперимент с CatBoost добавляет цену нефти и календарные признаки праздников. Цена нефти приведена к ежедневной частоте, пропуски заполнены по соседним доступным значениям. Праздники сопоставляются с магазинами с учётом уровня National, Regional и Local.

In [212]:
# Загружаем внешние временные данные

oil = pd.read_csv(
    DATA_PATH / 'oil.csv',
    parse_dates=['date']
)

holidays = pd.read_csv(
    DATA_PATH / 'holidays_events.csv',
    parse_dates=['date']
)

print('OIL')
print('Размер:', oil.shape)
print(
    'Период:',
    oil['date'].min().date(),
    '—',
    oil['date'].max().date()
)
print(
    'NaN в dcoilwtico:',
    oil['dcoilwtico'].isna().sum()
)

print('\nHOLIDAYS')
print('Размер:', holidays.shape)
print(
    'Период:',
    holidays['date'].min().date(),
    '—',
    holidays['date'].max().date()
)
print(
    'NaN всего:',
    holidays.isna().sum().sum()
)

print('\nТипы holiday:')
print(
    holidays['type'].value_counts()
)

print('\nLocale:')
print(
    holidays['locale'].value_counts()
)

OIL
Размер: (1218, 2)
Период: 2013-01-01 — 2017-08-31
NaN в dcoilwtico: 43

HOLIDAYS
Размер: (350, 6)
Период: 2012-03-02 — 2017-12-26
NaN всего: 0

Типы holiday:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

Locale:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64


In [213]:
# Подготавливаем ежедневную цену нефти
oil_daily = (
    oil
    .set_index('date')
    .sort_index()
    .asfreq('D')
)

print(
    'После перехода к ежедневной частоте:',
    oil_daily.shape
)

print(
    'NaN ДО заполнения:',
    oil_daily['dcoilwtico'].isna().sum()
)


# Заполняем отсутствующие календарные дни
oil_daily['oil_price'] = (
    oil_daily['dcoilwtico']
    .ffill()
    .bfill()
    .astype('float32')
)

oil_daily = (
    oil_daily[['oil_price']]
    .reset_index()
)


print(
    '\nПериод:',
    oil_daily['date'].min().date(),
    '—',
    oil_daily['date'].max().date()
)

print(
    'Количество календарных дней:',
    len(oil_daily)
)

print(
    'NaN ПОСЛЕ заполнения:',
    oil_daily['oil_price'].isna().sum()
)


# Проверяем именно нужный нам период
oil_check = oil_daily[
    (oil_daily['date'] >= '2017-07-25') &
    (oil_daily['date'] <= '2017-08-31')
]

display(oil_check.head(10))

После перехода к ежедневной частоте: (1704, 1)
NaN ДО заполнения: 529

Период: 2013-01-01 — 2017-08-31
Количество календарных дней: 1704
NaN ПОСЛЕ заполнения: 0


,date,oil_price
1666,2017-07-25,47.770000
1667,2017-07-26,48.580002
1668,2017-07-27,49.049999
1669,2017-07-28,49.720001
1670,2017-07-29,49.720001
1671,2017-07-30,49.720001
1672,2017-07-31,50.209999
1673,2017-08-01,49.189999
1674,2017-08-02,49.599998
1675,2017-08-03,49.029999


Для корректного сопоставления локальных и региональных праздников строим календарь date х store и добавляем город и регион магазина.

In [214]:
# Все календарные даты, которые нужны для ML
calendar_dates = pd.DataFrame({
    'date': pd.date_range(
        ML_DATA_START,
        ML_DATA_END,
        freq='D'
    )
})

# Декартово произведение: каждая дата × каждый магазин
store_calendar = (
    calendar_dates
    .merge(
        stores[
            ['store_nbr', 'city', 'state']
        ],
        how='cross'
    )
)

print(
    'Размер store_calendar:',
    store_calendar.shape
)

print(
    'Количество дат:',
    store_calendar['date'].nunique()
)

print(
    'Количество магазинов:',
    store_calendar['store_nbr'].nunique()
)

print(
    'Ожидаемое число строк:',
    len(calendar_dates) * stores['store_nbr'].nunique()
)

print('\nNaN:')
print(
    store_calendar.isna().sum()
)

display(store_calendar.head(10))

Размер store_calendar: (5778, 4)
Количество дат: 107
Количество магазинов: 54
Ожидаемое число строк: 5778

NaN:
date         0
store_nbr    0
city         0
state        0
dtype: int64


,date,store_nbr,city,state
0,2017-05-01,1,Quito,Pichincha
1,2017-05-01,2,Quito,Pichincha
2,2017-05-01,3,Quito,Pichincha
3,2017-05-01,4,Quito,Pichincha
4,2017-05-01,5,Santo Domingo,Santo Domingo de los Tsachilas
5,2017-05-01,6,Quito,Pichincha
6,2017-05-01,7,Quito,Pichincha
7,2017-05-01,8,Quito,Pichincha
8,2017-05-01,9,Quito,Pichincha
9,2017-05-01,10,Quito,Pichincha


Строим holiday-признаки с учётом географии.

Сделаем несколько отдельных бинарных признаков, а не один общий holiday. Это позволит CatBoost самостоятельно определить, какие типы событий действительно влияют на продажи.

Событие сначала должно относиться к конкретному магазину по National / Regional / Local, а затем мы определяем его тип. Для Holiday дополнительно учитываем transferred: исходная перенесённая дата не считается фактическим праздничным днём.

In [215]:
# Копия календаря магазинов
holiday_calendar = store_calendar.copy()

HOLIDAY_FEATURES = [
    'is_holiday',
    'is_event',
    'is_additional',
    'is_transfer',
    'is_bridge',
    'is_work_day'
]

for col in HOLIDAY_FEATURES:
    holiday_calendar[col] = np.int8(0)


# Обрабатываем события по одному
for _, event in holidays.iterrows():

    event_date = event['date']
    event_type = event['type']
    locale = event['locale']
    locale_name = event['locale_name']
    transferred = event['transferred']

    # Нас интересует только наш ML-период
    if (
        event_date < pd.Timestamp(ML_DATA_START)
        or event_date > pd.Timestamp(ML_DATA_END)
    ):
        continue

    # Определяем, каким магазинам относится событие
    mask = (
        holiday_calendar['date'] == event_date
    )

    if locale == 'Regional':
        mask &= (
            holiday_calendar['state'] == locale_name
        )

    elif locale == 'Local':
        mask &= (
            holiday_calendar['city'] == locale_name
        )

    # National: дополнительных ограничений нет

    # Тип события
    if event_type == 'Holiday':

        # Перенесённая исходная дата не считается фактическим holiday
        if not transferred:
            holiday_calendar.loc[
                mask,
                'is_holiday'
            ] = 1

    elif event_type == 'Event':
        holiday_calendar.loc[
            mask,
            'is_event'
        ] = 1

    elif event_type == 'Additional':
        holiday_calendar.loc[
            mask,
            'is_additional'
        ] = 1

    elif event_type == 'Transfer':
        holiday_calendar.loc[
            mask,
            'is_transfer'
        ] = 1

    elif event_type == 'Bridge':
        holiday_calendar.loc[
            mask,
            'is_bridge'
        ] = 1

    elif event_type == 'Work Day':
        holiday_calendar.loc[
            mask,
            'is_work_day'
        ] = 1



# Проверяем результат
print('Размер:', holiday_calendar.shape)

print('\nКоличество единиц:')
print(
    holiday_calendar[
        HOLIDAY_FEATURES
    ].sum()
)

print('\nNaN:')
print(
    holiday_calendar[
        HOLIDAY_FEATURES
    ].isna().sum()
)

print('\nДаты, где есть хотя бы одно событие:')

event_rows = holiday_calendar[
    holiday_calendar[
        HOLIDAY_FEATURES
    ].sum(axis=1) > 0
]

display(
    event_rows[
        [
            'date',
            'store_nbr',
            'city',
            'state'
        ] + HOLIDAY_FEATURES
    ].head(20)
)

Размер: (5778, 10)

Количество единиц:
is_holiday        68
is_event          54
is_additional     70
is_transfer      108
is_bridge          0
is_work_day        0
dtype: int64

NaN:
is_holiday       0
is_event         0
is_additional    0
is_transfer      0
is_bridge        0
is_work_day      0
dtype: int64

Даты, где есть хотя бы одно событие:


,date,store_nbr,city,state,is_holiday,is_event,is_additional,is_transfer,is_bridge,is_work_day
0,2017-05-01,1,Quito,Pichincha,1,0,0,0,0,0
1,2017-05-01,2,Quito,Pichincha,1,0,0,0,0,0
2,2017-05-01,3,Quito,Pichincha,1,0,0,0,0,0
3,2017-05-01,4,Quito,Pichincha,1,0,0,0,0,0
4,2017-05-01,5,Santo Domingo,Santo Domingo de los Tsachilas,1,0,0,0,0,0
5,2017-05-01,6,Quito,Pichincha,1,0,0,0,0,0
6,2017-05-01,7,Quito,Pichincha,1,0,0,0,0,0
7,2017-05-01,8,Quito,Pichincha,1,0,0,0,0,0
8,2017-05-01,9,Quito,Pichincha,1,0,0,0,0,0
9,2017-05-01,10,Quito,Pichincha,1,0,0,0,0,0


В нашем окне 01.05–15.08.2017 получили 68 применений обычных праздников, 54 событий, 70 дополнительных праздничных дней и 108 переносов. Bridge и Work Day в этом периоде отсутствуют. На 1 мая видно национальный праздник: is_holiday=1 у магазинов в разных городах и штатах, как и должно быть.

Теперь можно собрать CatBoost v5 с внешними признаками. Добавляем oil + holidays в train и validation

In [216]:
# Таблица внешних признаков date х store
external_features = holiday_calendar[
    [
        'date',
        'store_nbr',
        'is_holiday',
        'is_event',
        'is_additional',
        'is_transfer',
        'is_bridge',
        'is_work_day'
    ]
].copy()

# Добавляем цену нефти по дате
external_features = external_features.merge(
    oil_daily,
    on='date',
    how='left'
)

print('Размер external_features:', external_features.shape)

print('\nNaN во внешних признаках:')
print(
    external_features[
        [
            'oil_price',
            'is_holiday',
            'is_event',
            'is_additional',
            'is_transfer',
            'is_bridge',
            'is_work_day'
        ]
    ].isna().sum()
)


# Добавляем external features к ML-таблицам
ml_train_v5 = ml_train_v3.merge(
    external_features,
    on=['date', 'store_nbr'],
    how='left'
)

ml_val_v5 = ml_val_v3.merge(
    external_features,
    on=['date', 'store_nbr'],
    how='left'
)


EXTERNAL_FEATURES = [
    'oil_price',
    'is_holiday',
    'is_event',
    'is_additional',
    'is_transfer',
    'is_bridge',
    'is_work_day'
]


print('\nTRAIN:', ml_train_v5.shape)
print('VALIDATION:', ml_val_v5.shape)

print('\nNaN TRAIN:')
print(
    ml_train_v5[
        EXTERNAL_FEATURES
    ].isna().sum()
)

print('\nNaN VALIDATION:')
print(
    ml_val_v5[
        EXTERNAL_FEATURES
    ].isna().sum()
)


print('\nПример внешних признаков:')
display(
    ml_train_v5[
        [
            'date',
            'store_nbr',
            'item_nbr',
            'target'
        ] + EXTERNAL_FEATURES
    ].head(15)
)

Размер external_features: (5778, 9)

NaN во внешних признаках:
oil_price        0
is_holiday       0
is_event         0
is_additional    0
is_transfer      0
is_bridge        0
is_work_day      0
dtype: int64

TRAIN: (2366720, 36)
VALIDATION: (2358096, 37)

NaN TRAIN:
oil_price        0
is_holiday       0
is_event         0
is_additional    0
is_transfer      0
is_bridge        0
is_work_day      0
dtype: int64

NaN VALIDATION:
oil_price        0
is_holiday       0
is_event         0
is_additional    0
is_transfer      0
is_bridge        0
is_work_day      0
dtype: int64

Пример внешних признаков:


,date,store_nbr,item_nbr,target,oil_price,is_holiday,is_event,is_additional,is_transfer,is_bridge,is_work_day
0,2017-07-15,1,96995,0.0,46.529999,0,0,0,0,0,0
1,2017-07-16,1,96995,0.0,46.529999,0,0,0,0,0,0
2,2017-07-17,1,96995,0.0,46.020000,0,0,0,0,0,0
3,2017-07-18,1,96995,0.0,46.400002,0,0,0,0,0,0
4,2017-07-19,1,96995,0.0,47.099998,0,0,0,0,0,0
5,2017-07-20,1,96995,0.0,46.730000,0,0,0,0,0,0
6,2017-07-21,1,96995,3.0,45.779999,0,0,0,0,0,0
7,2017-07-22,1,96995,2.0,45.779999,0,0,0,0,0,0
8,2017-07-23,1,96995,0.0,45.779999,0,0,0,0,0,0
9,2017-07-24,1,96995,0.0,46.209999,0,0,0,0,0,0


Готовим X_train_v5 и X_val_v5. Используем пять информативных внешних признаков: oil_price, is_holiday, is_event, is_additional, is_transfer

In [217]:
# Информативные внешние признаки
EXTERNAL_MODEL_FEATURES = [
    'oil_price',
    'is_holiday',
    'is_event',
    'is_additional',
    'is_transfer'
]


# Полный набор признаков CatBoost v5
CATBOOST_V5_FEATURES = (
    CATBOOST_V3_FEATURES
    + EXTERNAL_MODEL_FEATURES
)


X_train_v5 = ml_train_v5[
    CATBOOST_V5_FEATURES
]

X_val_v5 = ml_val_v5[
    CATBOOST_V5_FEATURES
]


# Проверки
print(
    'X_train_v5:',
    X_train_v5.shape
)

print(
    'X_val_v5:',
    X_val_v5.shape
)

print(
    '\nКоличество признаков:',
    len(CATBOOST_V5_FEATURES)
)

print('\nНовые внешние признаки:')
print(
    X_train_v5[
        EXTERNAL_MODEL_FEATURES
    ].dtypes
)

print(
    '\nПамять X_train_v5:',
    f'{X_train_v5.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    'Память X_val_v5:',
    f'{X_val_v5.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    '\nNaN train:',
    X_train_v5.isna().sum().sum()
)

print(
    'NaN validation:',
    X_val_v5.isna().sum().sum()
)

print('\nВсе признаки v5:')
print(CATBOOST_V5_FEATURES)

X_train_v5: (2366720, 25)
X_val_v5: (2358096, 25)

Количество признаков: 25

Новые внешние признаки:
oil_price        float32
is_holiday          int8
is_event            int8
is_additional       int8
is_transfer         int8
dtype: object

Память X_train_v5: 121.88 MB
Память X_val_v5: 121.44 MB

NaN train: 0
NaN validation: 0

Все признаки v5:
['horizon', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'onpromotion', 'family_code', 'class', 'perishable', 'city_code', 'state_code', 'type_code', 'cluster', 'oil_price', 'is_holiday', 'is_event', 'is_additional', 'is_transfer']


Обучаем CatBoost v5 с oil + holidays

In [218]:
# CatBoost v5
# v4 + oil + holiday/event features
catboost_v5 = CatBoostRegressor(
    iterations=500,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=25,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v5.fit(
    X_train_v5,
    y_train_log,

    # Те же веса, что использовали в v4
    sample_weight=train_weights_v4,

    eval_set=(
        X_val_v5,
        y_val_log
    ),

    early_stopping_rounds=30,
    use_best_model=True
)

catboost_v5_time = time.time() - start_time


print(
    f'\nВремя обучения CatBoost v5: '
    f'{catboost_v5_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v5.get_best_iteration()
)

print(
    'Лучший validation RMSE в log-шкале:',
    catboost_v5.get_best_score()['validation']['RMSE']
)

0:	learn: 1.0209052	test: 1.0099772	best: 1.0099772 (0)	total: 279ms	remaining: 2m 19s
25:	learn: 0.6664923	test: 0.6757305	best: 0.6757305 (25)	total: 6.21s	remaining: 1m 53s
50:	learn: 0.6503751	test: 0.6637539	best: 0.6637539 (50)	total: 12.1s	remaining: 1m 46s
75:	learn: 0.6468959	test: 0.6626129	best: 0.6626129 (75)	total: 17.8s	remaining: 1m 39s
100:	learn: 0.6447038	test: 0.6616257	best: 0.6616257 (100)	total: 23.5s	remaining: 1m 32s
125:	learn: 0.6428215	test: 0.6612477	best: 0.6612417 (124)	total: 29.3s	remaining: 1m 26s
150:	learn: 0.6413683	test: 0.6606174	best: 0.6606173 (147)	total: 35.1s	remaining: 1m 21s
175:	learn: 0.6400085	test: 0.6600506	best: 0.6600506 (175)	total: 41.1s	remaining: 1m 15s
200:	learn: 0.6388456	test: 0.6594746	best: 0.6594746 (200)	total: 46.9s	remaining: 1m 9s
225:	learn: 0.6377352	test: 0.6591045	best: 0.6591045 (225)	total: 52.7s	remaining: 1m 3s
250:	learn: 0.6366542	test: 0.6586237	best: 0.6586237 (250)	total: 58.6s	remaining: 58.1s
275:	learn: 

v5 по validation log-RMSE оказался хуже v4.

У v4 было 0.654568, а у v5 стало 0.656856. Также v5 остановился раньше - на 448-й итерации, то есть после добавления oil + holiday features дальнейшее обучение перестало улучшать validation.

Считаем MAE и NWRMSLE для CatBoost v5

In [219]:
# Прогноз CatBoost v5
catboost_v5_pred_log = catboost_v5.predict(
    X_val_v5
)

# Возвращаемся из log1p в исходную шкалу
catboost_v5_pred = np.expm1(
    catboost_v5_pred_log
)

print(
    'Минимальный прогноз ДО clipping:',
    catboost_v5_pred.min()
)

print(
    'Максимальный прогноз ДО clipping:',
    catboost_v5_pred.max()
)

# Отрицательные прогнозы заменяем на 0
catboost_v5_pred = np.clip(
    catboost_v5_pred,
    0,
    None
)


# MAE
catboost_v5_mae = np.mean(
    np.abs(
        ml_val_eval['target'].values
        - catboost_v5_pred
    )
)


# NWRMSLE
catboost_v5_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(catboost_v5_pred)
            -
            np.log1p(
                ml_val_eval['target'].values
            )
        ) ** 2,
        weights=ml_val_eval['weight'].values
    )
)


print(
    f'\nCatBoost v5 MAE:     '
    f'{catboost_v5_mae:.4f}'
)

print(
    f'CatBoost v5 NWRMSLE: '
    f'{catboost_v5_nwrmsle:.4f}'
)


# Полная история CatBoost-экспериментов
catboost_all_results = pd.DataFrame({
    'Model': [
        'CatBoost v1 — raw target',
        'CatBoost v2 — log1p target',
        'CatBoost v3 — + promo + metadata',
        'CatBoost v4 — + perishable weights',
        'CatBoost v5 — + oil + holidays'
    ],

    'MAE': [
        catboost_v1_mae,
        catboost_v2_mae,
        catboost_v3_mae,
        catboost_v4_mae,
        catboost_v5_mae
    ],

    'NWRMSLE': [
        catboost_v1_nwrmsle,
        catboost_v2_nwrmsle,
        catboost_v3_nwrmsle,
        catboost_v4_nwrmsle,
        catboost_v5_nwrmsle
    ]
})

display(
    catboost_all_results.sort_values(
        'NWRMSLE'
    )
)

Минимальный прогноз ДО clipping: -0.4594282346103248
Максимальный прогноз ДО clipping: 525.4236520879768

CatBoost v5 MAE:     3.1303
CatBoost v5 NWRMSLE: 0.6570


,Model,MAE,NWRMSLE
3,CatBoost v4 — + perishable weights,3.111792,0.656179
2,CatBoost v3 — + promo + metadata,3.112142,0.656276
4,CatBoost v5 — + oil + holidays,3.130275,0.657016
1,CatBoost v2 — log1p target,3.209863,0.679722
0,CatBoost v1 — raw target,3.579256,0.820891


CatBoost v5 не улучшил качество на validation: NWRMSLE увеличился с 0.6562 для v4 до 0.6570 для v5, а MAE — с 3.1118 до 3.1303. Поэтому внешние признаки oil и holidays в текущей конфигурации не используются в финальной модели.

### 6.7 Выбор финальной конфигурации

Сравним результаты последовательных экспериментов CatBoost, проведённых на одном training origin (`train_end = 2017-07-14`) и одном validation origin (`train_end = 2017-07-30`).

In [239]:
catboost_ablation = pd.DataFrame({
    'Model': [
        'v1: raw target',
        'v2: log1p target',
        'v3: + promotion & metadata',
        'v4: + perishable weights',
        'v5: + oil & holidays'
    ],
    'MAE': [
        3.579256,
        3.209863,
        3.112142,
        3.111792,
        3.130275
    ],
    'NWRMSLE': [
        0.820891,
        0.679722,
        0.656276,
        0.656179,
        0.657016
    ]
})

display(catboost_ablation)

,Model,MAE,NWRMSLE
0,v1: raw target,3.579256,0.820891
1,v2: log1p target,3.209863,0.679722
2,v3: + promotion & metadata,3.112142,0.656276
3,v4: + perishable weights,3.111792,0.656179
4,v5: + oil & holidays,3.130275,0.657016


Наиболее заметное улучшение получено при переходе к log1p(target). Добавление onpromotion и метаданных товаров и магазинов также улучшило качество, а использование весов для скоропортящихся товаров дало небольшое дополнительное снижение ошибки.

Внешние признаки в виде цены нефти и календаря праздников на выбранном validation-периоде улучшения не дали. Поэтому для финального обучения выбрана конфигурация v4.

In [221]:
# Освобождаем память после ablation-экспериментов перед финальным обучением

objects_to_delete = [
    'X_train_v1', 'X_val_v1',
    'X_train_v3', 'X_val_v3',
    'X_train_v5', 'X_val_v5',
    'ml_train_v3', 'ml_val_v3',
    'ml_train_v5', 'ml_val_v5',
    'promo_data',
    'external_features',
    'store_calendar',
    'holiday_calendar'
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

gc.collect()

print('Память после ablation-экспериментов освобождена.')

Память после ablation-экспериментов освобождена.


### 6.8 Финальное обучение CatBoost v4 на нескольких historical origins

После выбора конфигурации v4 увеличиваем обучающую выборку с помощью нескольких historical forecast origins. Используются три точки окончания доступной истории: 19.06.2017, 03.07.2017 и 14.07.2017. Для каждого origin строится отдельная 16-дневная supervised-выборка по той же схеме признаков.

Validation остаётся неизменной: train_end = 2017-07-30, target-период - 31.07.2017–15.08.2017.

In [14]:
ML_TRAIN_ENDS = [
    pd.Timestamp('2017-06-19'),
    pd.Timestamp('2017-07-03'),
    pd.Timestamp('2017-07-14')
]

for train_end in ML_TRAIN_ENDS:
    forecast_start = train_end + pd.Timedelta(days=1)
    forecast_end = train_end + pd.Timedelta(days=FORECAST_HORIZON)

    print(
        f'Origin {train_end.date()} -> '
        f'target {forecast_start.date()} — {forecast_end.date()}'
    )

Origin 2017-06-19 -> target 2017-06-20 — 2017-07-05
Origin 2017-07-03 -> target 2017-07-04 — 2017-07-19
Origin 2017-07-14 -> target 2017-07-15 — 2017-07-30


In [15]:
ml_train_1 = build_ml_origin(
    ml_data=ml_data,
    train_end=pd.Timestamp('2017-06-19'),
    forecast_horizon=FORECAST_HORIZON,
    active_days=16
)

print('ORIGIN 1')
print('Размер:', ml_train_1.shape)
print(
    'Период target:',
    ml_train_1['date'].min().date(),
    '—',
    ml_train_1['date'].max().date()
)
print(
    'Пар:',
    ml_train_1[['store_nbr', 'item_nbr']]
    .drop_duplicates()
    .shape[0]
)
print(
    'Память:',
    f'{ml_train_1.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

ORIGIN 1
Размер: (2370064, 17)
Период target: 2017-06-20 — 2017-07-05
Пар: 148129
Память: 133.36 MB


In [16]:
ml_train_2 = build_ml_origin(
    ml_data=ml_data,
    train_end=pd.Timestamp('2017-07-03'),
    forecast_horizon=FORECAST_HORIZON,
    active_days=16
)

print('ORIGIN 2')
print('Размер:', ml_train_2.shape)
print(
    'Период target:',
    ml_train_2['date'].min().date(),
    '—',
    ml_train_2['date'].max().date()
)
print(
    'Пар:',
    ml_train_2[['store_nbr', 'item_nbr']]
    .drop_duplicates()
    .shape[0]
)
print(
    'Память:',
    f'{ml_train_2.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

ORIGIN 2
Размер: (2370096, 17)
Период target: 2017-07-04 — 2017-07-19
Пар: 148131
Память: 133.36 MB


In [17]:
def prepare_v4_features(origin_df, ml_data, items, stores):
    """
    Добавляет признаки, используемые в CatBoost v4:
    promotion, item metadata и store metadata.
    """

    # Promotion
    promo_data = ml_data[
        [
            'date',
            'store_nbr',
            'item_nbr',
            'onpromotion'
        ]
    ]

    result = origin_df.merge(
        promo_data,
        on=['date', 'store_nbr', 'item_nbr'],
        how='left'
    )

    result['onpromotion'] = (
        result['onpromotion']
        .fillna(False)
        .astype('int8')
    )

    # Item metadata
    item_features = items[
        [
            'item_nbr',
            'family',
            'class',
            'perishable'
        ]
    ]

    result = result.merge(
        item_features,
        on='item_nbr',
        how='left'
    )

    # Store metadata
    store_features = stores[
        [
            'store_nbr',
            'city',
            'state',
            'type',
            'cluster'
        ]
    ]

    result = result.merge(
        store_features,
        on='store_nbr',
        how='left'
    )

    return result

In [18]:
ml_train_1_v4 = prepare_v4_features(
    ml_train_1,
    ml_data,
    items,
    stores
)

print('ORIGIN 1 + V4 FEATURES')
print('Размер:', ml_train_1_v4.shape)

check_cols = [
    'onpromotion',
    'family',
    'class',
    'perishable',
    'city',
    'state',
    'type',
    'cluster'
]

print('\nNaN:')
print(
    ml_train_1_v4[check_cols]
    .isna()
    .sum()
)

print(
    '\nПамять:',
    f'{ml_train_1_v4.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

/tmp/ipykernel_58/3289998330.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


ORIGIN 1 + V4 FEATURES
Размер: (2370064, 25)

NaN:
onpromotion    0
family         0
class          0
perishable     0
city           0
state          0
type           0
cluster        0
dtype: int64

Память: 645.52 MB


In [23]:
# Единые словари кодирования категорий
family_map = {
    value: i
    for i, value in enumerate(items['family'].unique())
}

city_map = {
    value: i
    for i, value in enumerate(stores['city'].unique())
}

state_map = {
    value: i
    for i, value in enumerate(stores['state'].unique())
}

type_map = {
    value: i
    for i, value in enumerate(stores['type'].unique())
}


def encode_v4_categories(df):
    df['family_code'] = (
        df['family']
        .map(family_map)
        .astype('int8')
    )

    df['city_code'] = (
        df['city']
        .map(city_map)
        .astype('int8')
    )

    df['state_code'] = (
        df['state']
        .map(state_map)
        .astype('int8')
    )

    df['type_code'] = (
        df['type']
        .map(type_map)
        .astype('int8')
    )

    # Числовые metadata тоже уменьшаем
    df['class'] = df['class'].astype('int16')
    df['perishable'] = df['perishable'].astype('int8')
    df['cluster'] = df['cluster'].astype('int8')

    # Строковые версии больше не нужны
    df.drop(
        columns=['family', 'city', 'state', 'type'],
        inplace=True
    )

    return df

In [24]:
ml_train_1_v4 = encode_v4_categories(
    ml_train_1_v4
)

gc.collect()

print('Размер:', ml_train_1_v4.shape)

print(
    'Память после кодирования:',
    f'{ml_train_1_v4.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print('\nТипы metadata:')
print(
    ml_train_1_v4[
        [
            'onpromotion',
            'family_code',
            'class',
            'perishable',
            'city_code',
            'state_code',
            'type_code',
            'cluster'
        ]
    ].dtypes
)

print('\nNaN:')
print(
    ml_train_1_v4[
        [
            'onpromotion',
            'family_code',
            'class',
            'perishable',
            'city_code',
            'state_code',
            'type_code',
            'cluster'
        ]
    ].isna().sum()
)

Размер: (2370064, 25)
Память после кодирования: 153.70 MB

Типы metadata:
onpromotion     int8
family_code     int8
class          int16
perishable      int8
city_code       int8
state_code      int8
type_code       int8
cluster         int8
dtype: object

NaN:
onpromotion    0
family_code    0
class          0
perishable     0
city_code      0
state_code     0
type_code      0
cluster        0
dtype: int64


In [25]:
ml_train_2_v4 = prepare_v4_features(
    ml_train_2,
    ml_data,
    items,
    stores
)

ml_train_2_v4 = encode_v4_categories(
    ml_train_2_v4
)

gc.collect()

print('ORIGIN 2 + V4 FEATURES')
print('Размер:', ml_train_2_v4.shape)

print(
    'Память:',
    f'{ml_train_2_v4.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    'NaN в признаках:',
    ml_train_2_v4[
        [
            'onpromotion',
            'family_code',
            'class',
            'perishable',
            'city_code',
            'state_code',
            'type_code',
            'cluster'
        ]
    ].isna().sum().sum()
)

/tmp/ipykernel_58/3289998330.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


ORIGIN 2 + V4 FEATURES
Размер: (2370096, 25)
Память: 153.70 MB
NaN в признаках: 0


In [26]:
del ml_train_1
del ml_train_2

gc.collect()

print('Исходные origin 1 и origin 2 удалены из памяти.')

Исходные origin 1 и origin 2 удалены из памяти.


In [27]:
ml_train_3_v4 = prepare_v4_features(
    ml_train,
    ml_data,
    items,
    stores
)

ml_train_3_v4 = encode_v4_categories(
    ml_train_3_v4
)

gc.collect()

print('ORIGIN 3 + V4 FEATURES')
print('Размер:', ml_train_3_v4.shape)

print(
    'Период target:',
    ml_train_3_v4['date'].min().date(),
    '—',
    ml_train_3_v4['date'].max().date()
)

print(
    'Память:',
    f'{ml_train_3_v4.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print(
    'NaN в признаках:',
    ml_train_3_v4[
        [
            'onpromotion',
            'family_code',
            'class',
            'perishable',
            'city_code',
            'state_code',
            'type_code',
            'cluster'
        ]
    ].isna().sum().sum()
)

/tmp/ipykernel_58/3289998330.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


ORIGIN 3 + V4 FEATURES
Размер: (2366720, 25)
Период target: 2017-07-15 — 2017-07-30
Память: 153.48 MB
NaN в признаках: 0


In [28]:
ml_val_v4_final = prepare_v4_features(
    ml_val,
    ml_data,
    items,
    stores
)

ml_val_v4_final = encode_v4_categories(
    ml_val_v4_final
)

gc.collect()

print('VALIDATION + V4 FEATURES')
print('Размер:', ml_val_v4_final.shape)

print(
    'Период target:',
    ml_val_v4_final['date'].min().date(),
    '—',
    ml_val_v4_final['date'].max().date()
)

print(
    'Пар:',
    ml_val_v4_final[
        ['store_nbr', 'item_nbr']
    ].drop_duplicates().shape[0]
)

print(
    'Память:',
    f'{ml_val_v4_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

V4_META_FEATURES = [
    'onpromotion',
    'family_code',
    'class',
    'perishable',
    'city_code',
    'state_code',
    'type_code',
    'cluster'
]

print(
    'NaN в признаках:',
    ml_val_v4_final[
        V4_META_FEATURES
    ].isna().sum().sum()
)

/tmp/ipykernel_58/3289998330.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


VALIDATION + V4 FEATURES
Размер: (2358096, 25)
Период target: 2017-07-31 — 2017-08-15
Пар: 147381
Память: 152.92 MB
NaN в признаках: 0


In [29]:
FINAL_V4_FEATURES = [
    'horizon',
    'day_of_week',
    'day_of_month',
    'month',
    'is_weekend',

    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',

    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_28',

    'onpromotion',
    'family_code',
    'class',
    'perishable',
    'city_code',
    'state_code',
    'type_code',
    'cluster'
]

# Объединяем только признаки, которые реально нужны модели
X_train_final = pd.concat(
    [
        ml_train_1_v4[FINAL_V4_FEATURES],
        ml_train_2_v4[FINAL_V4_FEATURES],
        ml_train_3_v4[FINAL_V4_FEATURES]
    ],
    ignore_index=True
)

# Target сразу переводим в log1p
y_train_final = np.concatenate([
    np.log1p(ml_train_1_v4['target'].to_numpy(dtype='float32')),
    np.log1p(ml_train_2_v4['target'].to_numpy(dtype='float32')),
    np.log1p(ml_train_3_v4['target'].to_numpy(dtype='float32'))
]).astype('float32')

# Competition weights
train_weights_final = np.where(
    X_train_final['perishable'].to_numpy() == 1,
    1.25,
    1.0
).astype('float32')


# Validation
X_val_final = ml_val_v4_final[
    FINAL_V4_FEATURES
].copy()

y_val_final = np.log1p(
    ml_val_v4_final['target'].to_numpy(dtype='float32')
).astype('float32')

val_weights_final = np.where(
    X_val_final['perishable'].to_numpy() == 1,
    1.25,
    1.0
).astype('float32')


print('TRAIN')
print('X:', X_train_final.shape)
print('y:', y_train_final.shape)
print('weights:', train_weights_final.shape)

print(
    'Память X:',
    f'{X_train_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print('\nVALIDATION')
print('X:', X_val_final.shape)
print('y:', y_val_final.shape)
print('weights:', val_weights_final.shape)

print(
    'Память X:',
    f'{X_val_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
)

print('\nCHECK')
print('NaN train:', X_train_final.isna().sum().sum())
print('NaN validation:', X_val_final.isna().sum().sum())
print(
    'Train rows correct:',
    len(X_train_final)
    == (
        len(ml_train_1_v4)
        + len(ml_train_2_v4)
        + len(ml_train_3_v4)
    )
)

TRAIN
X: (7106880, 20)
y: (7106880,)
weights: (7106880,)
Память X: 284.66 MB

VALIDATION
X: (2358096, 20)
y: (2358096,)
weights: (2358096,)
Память X: 94.45 MB

CHECK
NaN train: 0
NaN validation: 0
Train rows correct: True


In [234]:
del ml_train_1_v4
del ml_train_2_v4
del ml_train_3_v4

# ml_train и ml_val тоже больше не нужны для обучения final v4
del ml_train
del ml_val

gc.collect()

print('Промежуточные training/validation DataFrame удалены.')
print('Финальные X/y/weights сохранены.')

Промежуточные training/validation DataFrame удалены.
Финальные X/y/weights сохранены.


In [52]:
from catboost import CatBoostRegressor
import time

catboost_v4_final = CatBoostRegressor(
    iterations=600,
    depth=7,
    learning_rate=0.08,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
    thread_count=-1
)

start_time = time.time()

catboost_v4_final.fit(
    X_train_final,
    y_train_final,
    sample_weight=train_weights_final,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=50,
    use_best_model=True
)

catboost_v4_final_time = time.time() - start_time

print(
    f'\nВремя обучения: '
    f'{catboost_v4_final_time / 60:.2f} минут'
)

print(
    'Лучшая итерация:',
    catboost_v4_final.get_best_iteration()
)

print(
    'Лучший validation RMSE:',
    catboost_v4_final
    .get_best_score()['validation']['RMSE']
)

0:	learn: 1.0196561	test: 1.0103742	best: 1.0103742 (0)	total: 749ms	remaining: 7m 28s
50:	learn: 0.6521812	test: 0.6594514	best: 0.6594514 (50)	total: 30.2s	remaining: 5m 25s
100:	learn: 0.6466576	test: 0.6554835	best: 0.6554835 (100)	total: 59s	remaining: 4m 51s
150:	learn: 0.6438580	test: 0.6533838	best: 0.6533838 (150)	total: 1m 28s	remaining: 4m 21s
200:	learn: 0.6418447	test: 0.6519343	best: 0.6519343 (200)	total: 1m 57s	remaining: 3m 52s
250:	learn: 0.6402437	test: 0.6507443	best: 0.6507443 (250)	total: 2m 26s	remaining: 3m 23s
300:	learn: 0.6387855	test: 0.6498869	best: 0.6498869 (300)	total: 2m 55s	remaining: 2m 54s
350:	learn: 0.6376629	test: 0.6492071	best: 0.6492071 (350)	total: 3m 24s	remaining: 2m 25s
400:	learn: 0.6366431	test: 0.6486822	best: 0.6486822 (400)	total: 3m 54s	remaining: 1m 56s
450:	learn: 0.6357451	test: 0.6482594	best: 0.6482594 (450)	total: 4m 23s	remaining: 1m 26s
500:	learn: 0.6349787	test: 0.6479124	best: 0.6479124 (500)	total: 4m 53s	remaining: 57.9s


In [236]:
# Прогноз финального CatBoost
catboost_v4_final_pred_log = catboost_v4_final.predict(
    X_val_final
)

# Возвращаем прогноз в исходную шкалу
catboost_v4_final_pred = np.expm1(
    catboost_v4_final_pred_log
)

# Отрицательные прогнозы заменяем на 0
catboost_v4_final_pred = np.clip(
    catboost_v4_final_pred,
    0,
    None
)

# Фактические значения validation
y_val_actual = ml_val_v4_final[
    'target'
].to_numpy(dtype='float32')


# MAE
catboost_v4_final_mae = np.mean(
    np.abs(
        y_val_actual
        - catboost_v4_final_pred
    )
)


# NWRMSLE
catboost_v4_final_nwrmsle = np.sqrt(
    np.average(
        (
            np.log1p(catboost_v4_final_pred)
            - np.log1p(y_val_actual)
        ) ** 2,
        weights=val_weights_final
    )
)


print('FINAL CATBOOST V4')
print(
    f'MAE:     {catboost_v4_final_mae:.6f}'
)
print(
    f'NWRMSLE: {catboost_v4_final_nwrmsle:.6f}'
)

print(
    '\nДиапазон прогнозов:',
    f'{catboost_v4_final_pred.min():.4f}',
    '—',
    f'{catboost_v4_final_pred.max():.4f}'
)

print(
    'Количество прогнозов:',
    len(catboost_v4_final_pred)
)

FINAL CATBOOST V4
MAE:     3.001017
NWRMSLE: 0.647597

Диапазон прогнозов: 0.0000 — 607.1308
Количество прогнозов: 2358096


Финальный CatBoost v4 получил MAE = 3.0010 и NWRMSLE = 0.6476. На выбранном validation-периоде обучение той же конфигурации на трёх historical origins снизило MAE относительно v4 на одном origin с 3.0480 до 3.0010, а NWRMSLE — с 0.6546 до 0.6476.

CatBoost использовал RMSE в log1p-шкале для контроля качества во время обучения, а итоговая NWRMSLE рассчитана отдельно с весами 1.25 для скоропортящихся товаров и 1.0 для остальных. Модель достигла последней доступной итерации (best_iteration = 599 при iterations = 600), поэтому early stopping фактически не сработал.

In [237]:
from pathlib import Path

RESULTS_PATH = Path('/kaggle/working/results')
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

catboost_v4_final.save_model(
    RESULTS_PATH / 'catboost_v4_final.cbm'
)

print(
    'Модель сохранена:',
    RESULTS_PATH / 'catboost_v4_final.cbm'
)

Модель сохранена: /kaggle/working/results/catboost_v4_final.cbm


### 6.9 Итоги CatBoost

In [240]:
catboost_results = pd.DataFrame({
    'Model': [
        'CatBoost v1: raw target',
        'CatBoost v2: log1p target',
        'CatBoost v3: + promotion & metadata',
        'CatBoost v4: + competition weights',
        'CatBoost v5: + external features',
        'Final CatBoost v4: 3 training origins'
    ],
    'MAE': [
        3.579256,
        3.209863,
        3.112142,
        3.111792,
        3.130275,
        catboost_v4_final_mae
    ],
    'NWRMSLE': [
        0.820891,
        0.679722,
        0.656276,
        0.656179,
        0.657016,
        catboost_v4_final_nwrmsle
    ]
})

display(catboost_results)

,Model,MAE,NWRMSLE
0,CatBoost v1: raw target,3.579256,0.820891
1,CatBoost v2: log1p target,3.209863,0.679722
2,CatBoost v3: + promotion & metadata,3.112142,0.656276
3,CatBoost v4: + competition weights,3.111792,0.656179
4,CatBoost v5: + external features,3.130275,0.657016
5,Final CatBoost v4: 3 training origins,3.001017,0.647597


В серии экспериментов наибольшее улучшение связано с переходом к логарифмированному target. Promotion и metadata дали дополнительное улучшение, а использование весов скоропортящихся товаров позволило немного снизить NWRMSLE.

Внешние признаки oil и holidays в данном эксперименте качество не улучшили, поэтому финальной конфигурацией была выбрана v4. После выбора конфигурации CatBoost v4 был переобучен на трёх historical origins. Финальная модель получила MAE = 3.0010 и NWRMSLE = 0.6476 на validation-периоде 31.07.2017-15.08.2017.

## 7. Промежуточные результаты

В экспериментах с CatBoost наиболее сильное улучшение относительно первой версии дала работа с целевой переменной: переход от исходного target к log1p(target) заметно снизил обе ошибки. Добавление onpromotion и метаданных товаров и магазинов также улучшило качество, а использование весов для скоропортящихся товаров дало небольшой дополнительный прирост.

Эксперимент с внешними признаками — ценой нефти и календарём праздников - на выбранном validation-периоде качество не улучшил, поэтому эти признаки не вошли в финальную конфигурацию.

После выбора конфигурации v4 модель была переобучена на трёх исторических forecast origins. Финальный CatBoost получил MAE = 3.0010 и NWRMSLE = 0.6476 на validation-периоде 31.07.2017-15.08.2017.

Классические AutoETS и AutoTheta оценивались отдельно на фиксированной подвыборке из 5000 временных рядов из-за вычислительной стоимости покомпонентного обучения, поэтому их метрики не сравниваются напрямую с результатами на полной validation-панели.

## 8. Deep Learning

В качестве нейросетевой модели используем многослойный перцептрон (MLP). Модель обучается на той же временной постановке прогнозирования и оценивается на том же validation-периоде, что и CatBoost, что позволяет напрямую сравнить качество моделей.

В качестве целевой переменной используем log1p(unit_sales). Для оценки качества прогнозы преобразуются обратно в исходную шкалу, после чего рассчитываются MAE и NWRMSLE.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Используемое устройство:', device)

PyTorch: 2.10.0+cu128
CUDA доступна: True
GPU: Tesla T4
Используемое устройство: cuda


In [30]:
# Признаки для MLP
DL_FEATURES = FINAL_V4_FEATURES.copy()

# Разделяем числовые и категориальные признаки
DL_CATEGORICAL_FEATURES = [
    'family_code',
    'class',
    'city_code',
    'state_code',
    'type_code',
    'cluster'
]

DL_NUMERIC_FEATURES = [
    col for col in DL_FEATURES
    if col not in DL_CATEGORICAL_FEATURES
]

print('Всего признаков:', len(DL_FEATURES))
print('Числовых:', len(DL_NUMERIC_FEATURES))
print('Категориальных:', len(DL_CATEGORICAL_FEATURES))

print('\nЧисловые признаки:')
print(DL_NUMERIC_FEATURES)

print('\nКатегориальные признаки:')
print(DL_CATEGORICAL_FEATURES)

print('\nTrain:', X_train_final.shape)
print('Validation:', X_val_final.shape)

print('\nПропуски train:', X_train_final[DL_FEATURES].isna().sum().sum())
print('Пропуски validation:', X_val_final[DL_FEATURES].isna().sum().sum())

Всего признаков: 20
Числовых: 14
Категориальных: 6

Числовые признаки:
['horizon', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'onpromotion', 'perishable']

Категориальные признаки:
['family_code', 'class', 'city_code', 'state_code', 'type_code', 'cluster']

Train: (7106880, 20)
Validation: (2358096, 20)

Пропуски train: 0
Пропуски validation: 0


### 8.1 Подготовка данных для MLP

Числовые признаки стандартизируются с использованием среднего и стандартного отклонения, рассчитанных только на обучающей выборке. Это позволяет избежать использования информации из validation при подготовке признаков.

Категориальные признаки далее будут представлены с помощью обучаемых embedding-слоёв.

In [31]:
# Среднее и стандартное отклонение считаем только по train
dl_numeric_mean = (
    X_train_final[DL_NUMERIC_FEATURES]
    .mean()
    .astype('float32')
)

dl_numeric_std = (
    X_train_final[DL_NUMERIC_FEATURES]
    .std()
    .replace(0, 1)
    .astype('float32')
)

print('Параметры нормализации рассчитаны.')
print('Количество числовых признаков:', len(dl_numeric_mean))

print('\nСредние:')
print(dl_numeric_mean)

print('\nСтандартные отклонения:')
print(dl_numeric_std)

Параметры нормализации рассчитаны.
Количество числовых признаков: 14

Средние:
horizon             8.500000
day_of_week         2.979009
day_of_month       17.372562
month               6.770727
is_weekend          0.291627
lag_1               5.588318
lag_7               5.426128
lag_14              6.307720
lag_28              5.869159
rolling_mean_7      5.763048
rolling_mean_14     5.846305
rolling_mean_28     5.805723
onpromotion         0.092784
perishable          0.222423
dtype: float32

Стандартные отклонения:
horizon             4.609773
day_of_week         1.994635
day_of_month        8.472744
month               0.420366
is_weekend          0.454512
lag_1              19.307388
lag_7              21.305620
lag_14             21.150743
lag_28             17.369722
rolling_mean_7     14.761628
rolling_mean_14    14.353848
rolling_mean_28    13.625066
onpromotion         0.290130
perishable          0.415874
dtype: float32


In [32]:
class FavoritaMLPDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        X,
        y,
        weights,
        numeric_features,
        categorical_features,
        numeric_mean,
        numeric_std
    ):
        # Сохраняем numpy-массивы
        self.X_num = X[numeric_features].to_numpy(dtype=np.float32)
        self.X_cat = X[categorical_features].to_numpy(dtype=np.int64)

        self.y = np.asarray(y, dtype=np.float32)
        self.weights = np.asarray(weights, dtype=np.float32)

        self.mean = numeric_mean[numeric_features].to_numpy(dtype=np.float32)
        self.std = numeric_std[numeric_features].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_num = (self.X_num[idx] - self.mean) / self.std

        return (
            torch.from_numpy(x_num),
            torch.from_numpy(self.X_cat[idx]),
            torch.tensor(self.y[idx], dtype=torch.float32),
            torch.tensor(self.weights[idx], dtype=torch.float32)
        )


train_dataset = FavoritaMLPDataset(
    X=X_train_final,
    y=y_train_final,
    weights=train_weights_final,
    numeric_features=DL_NUMERIC_FEATURES,
    categorical_features=DL_CATEGORICAL_FEATURES,
    numeric_mean=dl_numeric_mean,
    numeric_std=dl_numeric_std
)

val_dataset = FavoritaMLPDataset(
    X=X_val_final,
    y=y_val_final,
    weights=val_weights_final,
    numeric_features=DL_NUMERIC_FEATURES,
    categorical_features=DL_CATEGORICAL_FEATURES,
    numeric_mean=dl_numeric_mean,
    numeric_std=dl_numeric_std
)

print('Train dataset:', len(train_dataset))
print('Validation dataset:', len(val_dataset))

sample_num, sample_cat, sample_y, sample_weight = train_dataset[0]

print('\nОдин объект:')
print('Numeric:', sample_num.shape, sample_num.dtype)
print('Categorical:', sample_cat.shape, sample_cat.dtype)
print('Target:', sample_y.item())
print('Weight:', sample_weight.item())

Train dataset: 7106880
Validation dataset: 2358096

Один объект:
Numeric: torch.Size([14]) torch.float32
Categorical: torch.Size([6]) torch.int64
Target: 2.6390573978424072
Weight: 1.0


In [33]:
# Размер словаря для каждого категориального признака
category_sizes = {}

for col in DL_CATEGORICAL_FEATURES:
    max_train = int(X_train_final[col].max())
    max_val = int(X_val_final[col].max())

    # +1, так как коды начинаются с 0
    category_sizes[col] = max(max_train, max_val) + 1

print('Размеры категорий:')
for col, size in category_sizes.items():
    print(f'{col}: {size}')

Размеры категорий:
family_code: 33
class: 7781
city_code: 22
state_code: 16
type_code: 5
cluster: 18


### 8.2 Архитектура нейросетевой модели

Для категориальных признаков используются обучаемые embedding-представления. Полученные embeddings объединяются с нормализованными числовыми признаками и передаются в многослойный перцептрон.

Модель содержит два скрытых полносвязных слоя с ReLU-активацией и Dropout. Выходной слой формирует прогноз логарифмированной целевой переменной log1p(unit_sales).

In [34]:
class FavoritaMLP(nn.Module):
    def __init__(
        self,
        num_numeric_features,
        categorical_features,
        category_sizes
    ):
        super().__init__()

        self.categorical_features = categorical_features

        # Размер embedding для каждой категории
        self.embedding_dims = {
            col: min(32, max(4, int(np.ceil(np.sqrt(category_sizes[col])))))
            for col in categorical_features
        }

        self.embeddings = nn.ModuleDict({
            col: nn.Embedding(
                num_embeddings=category_sizes[col],
                embedding_dim=self.embedding_dims[col]
            )
            for col in categorical_features
        })

        total_embedding_dim = sum(self.embedding_dims.values())
        input_dim = num_numeric_features + total_embedding_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(64, 1)
        )

    def forward(self, x_num, x_cat):
        embedded = []

        for i, col in enumerate(self.categorical_features):
            embedded.append(
                self.embeddings[col](x_cat[:, i])
            )

        x_emb = torch.cat(embedded, dim=1)
        x = torch.cat([x_num, x_emb], dim=1)

        return self.mlp(x).squeeze(1)


model = FavoritaMLP(
    num_numeric_features=len(DL_NUMERIC_FEATURES),
    categorical_features=DL_CATEGORICAL_FEATURES,
    category_sizes=category_sizes
).to(device)

print(model)

print('\nEmbedding dimensions:')
for col, dim in model.embedding_dims.items():
    print(f'{col}: {dim}')

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f'\nОбучаемых параметров: {n_params:,}')

FavoritaMLP(
  (embeddings): ModuleDict(
    (family_code): Embedding(33, 6)
    (class): Embedding(7781, 32)
    (city_code): Embedding(22, 5)
    (state_code): Embedding(16, 4)
    (type_code): Embedding(5, 4)
    (cluster): Embedding(18, 5)
  )
  (mlp): Sequential(
    (0): Linear(in_features=70, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

Embedding dimensions:
family_code: 6
class: 32
city_code: 5
state_code: 4
type_code: 4
cluster: 5

Обучаемых параметров: 266,883


In [35]:
BATCH_SIZE = 4096

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

print('Train batches:', len(train_loader))
print('Validation batches:', len(val_loader))

# Проверяем один batch
x_num_batch, x_cat_batch, y_batch, w_batch = next(iter(train_loader))

print('\nДо переноса на GPU:')
print('Numeric:', x_num_batch.shape, x_num_batch.dtype)
print('Categorical:', x_cat_batch.shape, x_cat_batch.dtype)
print('Target:', y_batch.shape)
print('Weights:', w_batch.shape)

x_num_batch = x_num_batch.to(device, non_blocking=True)
x_cat_batch = x_cat_batch.to(device, non_blocking=True)

model.eval()

with torch.no_grad():
    pred_batch = model(x_num_batch, x_cat_batch)

print('\nПосле forward pass:')
print('Predictions:', pred_batch.shape)
print('Device:', pred_batch.device)
print('Min prediction:', pred_batch.min().item())
print('Max prediction:', pred_batch.max().item())

Train batches: 1736
Validation batches: 576

До переноса на GPU:
Numeric: torch.Size([4096, 14]) torch.float32
Categorical: torch.Size([4096, 6]) torch.int64
Target: torch.Size([4096])
Weights: torch.Size([4096])

После forward pass:
Predictions: torch.Size([4096])
Device: cuda:0
Min prediction: -0.38487762212753296
Max prediction: 0.303186297416687


In [40]:
def evaluate_mlp(model, loader, device):
    model.eval()

    weighted_squared_error = 0.0
    weight_sum = 0.0

    with torch.no_grad():
        for x_num, x_cat, y, weights in loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            weights = weights.to(device, non_blocking=True)

            pred = model(x_num, x_cat)

            weighted_squared_error += (
                weights * (pred - y) ** 2
            ).sum().item()

            weight_sum += weights.sum().item()

    return np.sqrt(weighted_squared_error / weight_sum)

In [41]:
import time
import copy

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

MAX_EPOCHS = 5
PATIENCE = 2

best_val_nwrmsle = np.inf
best_state = None
epochs_without_improvement = 0

history = []

for epoch in range(1, MAX_EPOCHS + 1):
    start_time = time.time()

    model.train()

    train_squared_error = 0.0
    train_weight_sum = 0.0

    for x_num, x_cat, y, weights in train_loader:
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        optimizer.zero_grad()

        pred = model(x_num, x_cat)

        loss = (
            weights * (pred - y) ** 2
        ).sum() / weights.sum()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        train_squared_error += (
            weights * (pred.detach() - y) ** 2
        ).sum().item()

        train_weight_sum += weights.sum().item()

    train_nwrmsle = np.sqrt(
        train_squared_error / train_weight_sum
    )

    val_nwrmsle = evaluate_mlp(
        model,
        val_loader,
        device
    )

    epoch_time = time.time() - start_time

    history.append({
        'epoch': epoch,
        'train_nwrmsle': train_nwrmsle,
        'val_nwrmsle': val_nwrmsle,
        'time_sec': epoch_time
    })

    print(
        f'Epoch {epoch:02d} | '
        f'Train NWRMSLE: {train_nwrmsle:.6f} | '
        f'Val NWRMSLE: {val_nwrmsle:.6f} | '
        f'Time: {epoch_time:.1f} sec'
    )

    if val_nwrmsle < best_val_nwrmsle:
        best_val_nwrmsle = val_nwrmsle
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print('\nEarly stopping.')
            break

# Возвращаем лучшую версию модели
model.load_state_dict(best_state)

print(
    f'\nBest validation NWRMSLE: '
    f'{best_val_nwrmsle:.6f}'
)

Epoch 01 | Train NWRMSLE: 0.711452 | Val NWRMSLE: 0.663743 | Time: 179.9 sec
Epoch 02 | Train NWRMSLE: 0.659339 | Val NWRMSLE: 0.660317 | Time: 180.2 sec
Epoch 03 | Train NWRMSLE: 0.651506 | Val NWRMSLE: 0.661064 | Time: 180.2 sec
Epoch 04 | Train NWRMSLE: 0.647349 | Val NWRMSLE: 0.665488 | Time: 179.4 sec

Early stopping.

Best validation NWRMSLE: 0.660317


### 8.3 Оценка качества MLP

После обучения восстанавливается состояние модели с минимальной ошибкой на validation. Прогнозы преобразуются из логарифмической шкалы обратно в исходную с помощью `expm1` и ограничиваются снизу нулём. Качество оценивается с помощью тех же MAE и NWRMSLE, что использовались для остальных моделей.

In [42]:
model.eval()

mlp_predictions_log = []

with torch.no_grad():
    for x_num, x_cat, _, _ in val_loader:
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True)

        pred_log = model(x_num, x_cat)

        mlp_predictions_log.append(
            pred_log.cpu().numpy()
        )

mlp_predictions_log = np.concatenate(
    mlp_predictions_log
).astype('float32')

# Возвращаем прогнозы в исходную шкалу
mlp_predictions = np.expm1(
    mlp_predictions_log
)

mlp_predictions = np.clip(
    mlp_predictions,
    0,
    None
)

# Фактические продажи в исходной шкале
mlp_actual = np.expm1(
    y_val_final.astype('float32')
)

# MAE
mlp_mae = np.mean(
    np.abs(mlp_actual - mlp_predictions)
)

# NWRMSLE
mlp_nwrmsle = np.sqrt(
    np.sum(
        val_weights_final *
        (
            np.log1p(mlp_predictions) -
            np.log1p(mlp_actual)
        ) ** 2
    )
    /
    np.sum(val_weights_final)
)

print('FINAL MLP')
print(f'MAE:      {mlp_mae:.6f}')
print(f'NWRMSLE:  {mlp_nwrmsle:.6f}')
print(
    f'Prediction range: '
    f'{mlp_predictions.min():.4f} — '
    f'{mlp_predictions.max():.4f}'
)
print('Count:', len(mlp_predictions))

FINAL MLP
MAE:      3.213383
NWRMSLE:  0.660317
Prediction range: 0.0000 — 6699.9155
Count: 2358096


### 8.4 Сравнение MLP и CatBoost

Для сравнения MLP и CatBoost используются одинаковый validation-период и одинаковые метрики. Это позволяет напрямую сопоставить качество ML- и DL-подходов.

In [44]:
catboost_v4_final_mae = 3.001017
catboost_v4_final_nwrmsle = 0.647597

final_model_comparison = pd.DataFrame({
    'Model': [
        'Final CatBoost v4',
        'MLP'
    ],
    'MAE': [
        catboost_v4_final_mae,
        mlp_mae
    ],
    'NWRMSLE': [
        catboost_v4_final_nwrmsle,
        mlp_nwrmsle
    ]
})

display(final_model_comparison)

,Model,MAE,NWRMSLE
0,Final CatBoost v4,3.001017,0.647597
1,MLP,3.213383,0.660317


MLP достиг MAE = 3.2134 и NWRMSLE = 0.6603. Лучший результат CatBoost составил MAE = 3.0010 и NWRMSLE = 0.6476. На выбранном validation-периоде CatBoost показал более высокое качество по обеим метрикам.

MLP достиг минимального validation NWRMSLE на второй эпохе, после чего качество перестало улучшаться, и обучение было остановлено с помощью early stopping. Полученный результат показывает, что применение нейросетевой модели само по себе не обеспечивает улучшения относительно градиентного бустинга на данной задаче. Одной из возможных причин является табличная структура используемых признаков, для которой CatBoost хорошо подходит. При этом более специализированные нейросетевые архитектуры для временных рядов и дополнительный подбор гиперпараметров потенциально могут дать другие результаты.

In [45]:
RESULTS_PATH = Path('/kaggle/working/results')
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'numeric_features': DL_NUMERIC_FEATURES,
    'categorical_features': DL_CATEGORICAL_FEATURES,
    'category_sizes': category_sizes,
    'numeric_mean': dl_numeric_mean.to_dict(),
    'numeric_std': dl_numeric_std.to_dict(),
    'best_val_nwrmsle': best_val_nwrmsle
}, RESULTS_PATH / 'mlp_final.pt')

print('MLP сохранён:', RESULTS_PATH / 'mlp_final.pt')

MLP сохранён: /kaggle/working/results/mlp_final.pt


## 9. Итоговое сравнение моделей

В ходе работы модели последовательно усложнялись: от простых baseline-прогнозов и классических методов временных рядов до градиентного бустинга и нейросетевой модели.

Основной метрикой выбрана NWRMSLE, учитывающая логарифмическую ошибку и повышенный вес скоропортящихся товаров. Дополнительно рассчитывается MAE в исходной шкале продаж.

Для CatBoost были проведены отдельные ablation-эксперименты с логарифмированием целевой переменной, признаками promotion и metadata, весами скоропортящихся товаров и внешними признаками. Финальная конфигурация CatBoost была дополнительно обучена на нескольких historical forecast origins.

In [46]:
final_results = pd.DataFrame({
    'Model': [
        'Naive',
        'SeasonalNaive',
        'CatBoost v1',
        'CatBoost v2',
        'CatBoost v3',
        'CatBoost v4',
        'CatBoost v5',
        'Final CatBoost v4 (3 origins)',
        'MLP'
    ],
    'MAE': [
        5.252054,
        3.967178,
        3.579256,
        3.209863,
        3.112142,
        3.111792,
        3.130275,
        3.001017,
        3.213383
    ],
    'NWRMSLE': [
        0.941351,
        0.879374,
        0.820891,
        0.679722,
        0.656276,
        0.656179,
        0.657016,
        0.647597,
        0.660317
    ]
})

final_results

,Model,MAE,NWRMSLE
0,Naive,5.252054,0.941351
1,SeasonalNaive,3.967178,0.879374
2,CatBoost v1,3.579256,0.820891
3,CatBoost v2,3.209863,0.679722
4,CatBoost v3,3.112142,0.656276
5,CatBoost v4,3.111792,0.656179
6,CatBoost v5,3.130275,0.657016
7,Final CatBoost v4 (3 origins),3.001017,0.647597
8,MLP,3.213383,0.660317


### 9.1 Основные результаты

Простые baseline-модели показали наиболее высокую ошибку. SeasonalNaive оказался лучше Naive, что подтверждает наличие выраженной недельной сезонности в данных.

Наиболее заметное улучшение CatBoost произошло после перехода от исходной целевой переменной к log1p(unit_sales): NWRMSLE снизился с 0.8209 до 0.6797. Добавление информации о promotion и metadata товаров и магазинов позволило дополнительно снизить NWRMSLE до 0.6563.

Использование весов скоропортящихся товаров практически не изменило MAE, но немного улучшило NWRMSLE. Добавление внешних признаков oil и holidays не привело к улучшению качества в выбранной конфигурации, поэтому они не были включены в финальную модель.

Лучший результат получен для CatBoost v4, обученного на трёх historical forecast origins: MAE = 3.0010 и NWRMSLE = 0.6476.

MLP показал MAE = 3.2134 и NWRMSLE = 0.6603. Таким образом, в проведённом эксперименте нейросетевая модель не превзошла CatBoost. Это показывает, что увеличение сложности модели само по себе не гарантирует улучшения качества. Для используемого набора преимущественно табличных признаков градиентный бустинг оказался более эффективным.

Классические модели AutoETS и AutoTheta оценивались отдельно на фиксированной выборке из 5000 временных рядов из-за вычислительных ограничений, поэтому их результаты не сравниваются напрямую с приведёнными выше метриками на полной validation-выборке.

## 10. Формирование Kaggle submission

Для формирования прогноза на test-период используется финальная конфигурация CatBoost v4. Модель была выбрана по результатам временной validation и обучена на нескольких historical forecast origins.

In [53]:
from pathlib import Path

RESULTS_PATH = Path('/kaggle/working/results')
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

CATBOOST_MODEL_PATH = RESULTS_PATH / 'catboost_v4_final.cbm'

catboost_v4_final.save_model(CATBOOST_MODEL_PATH)

print('Модель сохранена:', CATBOOST_MODEL_PATH)
print('Существует:', CATBOOST_MODEL_PATH.exists())
print(
    'Размер:',
    f'{CATBOOST_MODEL_PATH.stat().st_size / 1024**2:.2f} MB'
)

Модель сохранена: /kaggle/working/results/catboost_v4_final.cbm
Существует: True
Размер: 1.25 MB


In [55]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    '/kaggle/input/datasets/m2101119/'
    'favorita-grocery-sales-forecasting'
)

test_path = DATA_PATH / 'test.csv'

TEST_FORECAST_START = pd.Timestamp('2017-08-16')
TEST_FORECAST_END = pd.Timestamp('2017-08-31')
TEST_TRAIN_END = pd.Timestamp('2017-08-15')

print('test.csv существует:', test_path.exists())

test = pd.read_csv(
    test_path,
    dtype={
        'id': 'int32',
        'store_nbr': 'int16',
        'item_nbr': 'int32'
    },
    parse_dates=['date']
)

test['onpromotion'] = (
    test['onpromotion']
    .fillna(False)
    .astype(bool)
)

print('Test shape:', test.shape)
print(
    'Период:',
    test['date'].min().date(),
    '—',
    test['date'].max().date()
)
print('Дней:', test['date'].nunique())
print(
    'Уникальных пар:',
    test[['store_nbr', 'item_nbr']]
    .drop_duplicates()
    .shape[0]
)
print(
    'Пропуски onpromotion:',
    test['onpromotion'].isna().sum()
)

test.csv существует: True
Test shape: (3370464, 5)
Период: 2017-08-16 — 2017-08-31
Дней: 16
Уникальных пар: 210654
Пропуски onpromotion: 0


In [56]:
# Создаём test-таблицу в том же формате, что использовался для CatBoost

test_features = test[
    ['id', 'date', 'store_nbr', 'item_nbr', 'onpromotion']
].copy()

# Horizon: 1 соответствует 2017-08-16, ..., 16 — 2017-08-31
test_features['horizon'] = (
    test_features['date'] - TEST_TRAIN_END
).dt.days.astype('int8')

# Календарные признаки
test_features['day_of_week'] = (
    test_features['date'].dt.dayofweek.astype('int8')
)

test_features['day_of_month'] = (
    test_features['date'].dt.day.astype('int8')
)

test_features['month'] = (
    test_features['date'].dt.month.astype('int8')
)

test_features['is_weekend'] = (
    test_features['day_of_week'] >= 5
).astype('int8')

test_features['onpromotion'] = (
    test_features['onpromotion']
    .astype('int8')
)

# Исторические lag-признаки
for lag in [1, 7, 14, 28]:
    lag_date = TEST_TRAIN_END - pd.Timedelta(days=lag - 1)

    lag_values = (
        ml_data.loc[
            ml_data['date'] == lag_date,
            ['store_nbr', 'item_nbr', 'unit_sales_clean']
        ]
        .rename(columns={
            'unit_sales_clean': f'lag_{lag}'
        })
    )

    test_features = test_features.merge(
        lag_values,
        on=['store_nbr', 'item_nbr'],
        how='left'
    )

    test_features[f'lag_{lag}'] = (
        test_features[f'lag_{lag}']
        .fillna(0)
        .astype('float32')
    )

# Rolling mean
for window in [7, 14, 28]:
    window_start = (
        TEST_TRAIN_END -
        pd.Timedelta(days=window - 1)
    )

    rolling_values = (
        ml_data.loc[
            (ml_data['date'] >= window_start) &
            (ml_data['date'] <= TEST_TRAIN_END)
        ]
        .groupby(
            ['store_nbr', 'item_nbr'],
            as_index=False
        )['unit_sales_clean']
        .sum()
    )

    # В нашей sparse-постановке отсутствующие дни = 0,
    # поэтому делим сумму на полную длину окна
    rolling_values[f'rolling_mean_{window}'] = (
        rolling_values['unit_sales_clean'] / window
    ).astype('float32')

    rolling_values = rolling_values[
        [
            'store_nbr',
            'item_nbr',
            f'rolling_mean_{window}'
        ]
    ]

    test_features = test_features.merge(
        rolling_values,
        on=['store_nbr', 'item_nbr'],
        how='left'
    )

    test_features[f'rolling_mean_{window}'] = (
        test_features[f'rolling_mean_{window}']
        .fillna(0)
        .astype('float32')
    )

print('Test features:', test_features.shape)

print(
    'Пропуски:',
    test_features.isna().sum().sum()
)

print(
    'Horizon:',
    test_features['horizon'].min(),
    '—',
    test_features['horizon'].max()
)

display(test_features.head())

Test features: (3370464, 17)
Пропуски: 0
Horizon: 1 — 16


,id,date,store_nbr,item_nbr,onpromotion,horizon,day_of_week,day_of_month,month,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,125497040,2017-08-16,1,96995,0,1,2,16,8,0,0.0,0.0,1.0,0.0,0.142857,0.571429,0.535714
1,125497041,2017-08-16,1,99197,0,1,2,16,8,0,0.0,2.0,0.0,2.0,0.285714,0.357143,0.535714
2,125497042,2017-08-16,1,103501,0,1,2,16,8,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,125497043,2017-08-16,1,103520,0,1,2,16,8,0,0.0,0.0,1.0,1.0,0.857143,1.071429,1.357143
4,125497044,2017-08-16,1,103665,0,1,2,16,8,0,1.0,7.0,2.0,3.0,2.857143,2.714286,2.464286


In [57]:
# Добавляем metadata товаров
test_features = test_features.merge(
    items[
        ['item_nbr', 'family', 'class', 'perishable']
    ],
    on='item_nbr',
    how='left'
)

# Добавляем metadata магазинов
test_features = test_features.merge(
    stores[
        ['store_nbr', 'city', 'state', 'type', 'cluster']
    ],
    on='store_nbr',
    how='left'
)

# Кодируем категории теми же словарями,
# которые использовались при обучении CatBoost
test_features['family_code'] = (
    test_features['family']
    .map(family_map)
    .fillna(-1)
    .astype('int8')
)

test_features['city_code'] = (
    test_features['city']
    .map(city_map)
    .fillna(-1)
    .astype('int8')
)

test_features['state_code'] = (
    test_features['state']
    .map(state_map)
    .fillna(-1)
    .astype('int8')
)

test_features['type_code'] = (
    test_features['type']
    .map(type_map)
    .fillna(-1)
    .astype('int8')
)

test_features['class'] = (
    test_features['class']
    .fillna(-1)
    .astype('int16')
)

test_features['perishable'] = (
    test_features['perishable']
    .fillna(0)
    .astype('int8')
)

test_features['cluster'] = (
    test_features['cluster']
    .fillna(-1)
    .astype('int8')
)

# Оставляем признаки строго в том же порядке,
# что и при обучении финального CatBoost
X_test_final = test_features[
    FINAL_V4_FEATURES
].copy()

print('X_test_final:', X_test_final.shape)
print(
    'Количество признаков:',
    X_test_final.shape[1]
)
print(
    'Пропуски:',
    X_test_final.isna().sum().sum()
)

print(
    'Порядок признаков совпадает:',
    list(X_test_final.columns) == list(FINAL_V4_FEATURES)
)

print('\nПризнаки:')
print(list(X_test_final.columns))

X_test_final: (3370464, 20)
Количество признаков: 20
Пропуски: 0
Порядок признаков совпадает: True

Признаки:
['horizon', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'onpromotion', 'family_code', 'class', 'perishable', 'city_code', 'state_code', 'type_code', 'cluster']


In [58]:
# Прогноз в log1p-шкале
test_pred_log = catboost_v4_final.predict(
    X_test_final
)

# Возвращаемся в исходную шкалу продаж
test_pred = np.expm1(test_pred_log)

# Продажи не могут быть отрицательными
test_pred = np.clip(
    test_pred,
    0,
    None
).astype('float32')

print('Количество прогнозов:', len(test_pred))
print(
    'Диапазон прогнозов:',
    f'{test_pred.min():.4f} — {test_pred.max():.4f}'
)
print(
    'Средний прогноз:',
    f'{test_pred.mean():.4f}'
)
print(
    'NaN:',
    np.isnan(test_pred).sum()
)
print(
    'Inf:',
    np.isinf(test_pred).sum()
)

Количество прогнозов: 3370464
Диапазон прогнозов: 0.0000 — 621.0041
Средний прогноз: 2.7709
NaN: 0
Inf: 0


In [59]:
submission = pd.DataFrame({
    'id': test['id'].to_numpy(),
    'unit_sales': test_pred
})

submission_path = Path(
    '/kaggle/working/submission.csv'
)

submission.to_csv(
    submission_path,
    index=False
)

print('Submission сохранён:')
print(submission_path)

print('\nShape:', submission.shape)
print('Пропуски:', submission.isna().sum().sum())
print('Дубликаты id:', submission['id'].duplicated().sum())

print('\nПервые строки:')
display(submission.head())

print('\nПоследние строки:')
display(submission.tail())

Submission сохранён:
/kaggle/working/submission.csv

Shape: (3370464, 2)
Пропуски: 0
Дубликаты id: 0

Первые строки:


,id,unit_sales
0,125497040,0.326080
1,125497041,0.347001
2,125497042,0.094292
3,125497043,0.775997
4,125497044,1.760943



Последние строки:


,id,unit_sales
3370459,128867499,0.127366
3370460,128867500,0.076284
3370461,128867501,0.091166
3370462,128867502,0.094844
3370463,128867503,0.061035


### 10.1 Результат Kaggle submission

Финальный прогноз CatBoost был сформирован для test-периода 2017-08-16 — 2017-08-31 и отправлен в Kaggle.

Полученные результаты:

- Public Score: 0.53993
- Private Score: 0.54547

Submission успешно обработан Kaggle. Значения leaderboard score не сравниваются напрямую с validation NWRMSLE, поскольку они рассчитаны на другом временном периоде и другом наборе наблюдений.